# Fine-tuned Mistral vs RedSage Qwen (Kaggle)

Runs both models on the same attack paths with the same prompt, then scores them with the format metrics and the gemini-2.5-flash-lite judge. Generated by `create_notebook.py` from `evaluation/compare_models.py`.

Before *Run all*:
1. Settings: accelerator **GPU T4 x2**, internet **on**.
2. Input: attach the **finetuned** adapter dataset.
3. Add-ons > Secrets: add `OPENROUTER_API_KEY`. Without it you get format metrics only.
4. Rerunning after a disconnect skips everything already in `/kaggle/working/model_comparison`.

About 10 generations at 2 to 3 minutes each, plus about 30 GB of model downloads.

In [ ]:
!pip install -q -U "transformers>=5,<6" "peft>=0.18,<0.19" "bitsandbytes>=0.47,<0.50" "accelerate>=1.1"
print('Done.')

In [ ]:
# Ship the fixed repo modules into this runtime (source of truth).
import pathlib
pathlib.Path("compare_models.py").write_text("\"\"\"compare_models.py\n\nFine-tuned Mistral-7B against RedSage-Qwen3-8B on the same attack paths.\n\nBoth models get the same messages for every (path, prompt) pair, each rendered\nthrough its own chat template, and the same decoding settings and seed. The\nonly thing that changes between the two columns of the report is the model.\n\nThe models load one at a time, in 4-bit: bitsandbytes NF4 on an NVIDIA GPU,\nor MLX on Apple Silicon, where both models are converted from their official\nweights with the same settings and the adapter is translated to MLX's layout.\nEvery generation and every judge score is appended to disk as soon as it\nexists, so a disconnected session picks up where it stopped when rerun with\nthe same --out_dir.\n\nThe judge is gemini-2.5-flash-lite with the rubric from judge_eval.py, called\nthrough OpenRouter when OPENROUTER_API_KEY is set, otherwise through Gemini.\n\nRun from scenario_generation/:\n    python evaluation/compare_models.py --adapter_path fine_tuning/output\n    python evaluation/compare_models.py --backend mlx      # on a Mac\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE.parent))\nsys.path.insert(0, str(HERE))\n\nimport prompt_generator as pg  # noqa: E402  (also puts fine_tuning/ on sys.path)\nimport prompt_format  # noqa: E402\nimport manual_eval_metrics as metrics  # noqa: E402\nimport judge_eval as judge  # noqa: E402\n\n\nMISTRAL = \"mistral_ft\"\nREDSAGE = \"redsage\"\nLABELS = {MISTRAL: \"Fine-tuned Mistral\", REDSAGE: \"RedSage Qwen\"}\n\nREDSAGE_DEFAULT = \"RISys-Lab/RedSage-Qwen3-8B-Ins\"\nATTACK_PATHS_DEFAULT = HERE / \"inputs\" / \"attack_paths.json\"\n\n# Both entry routes, all four objectives, and the one two-hop path. The ranked\n# top five would all start from the same security group.\nSAMPLE_PATH_IDS = [\"AP-001\", \"AP-003\", \"AP-007\", \"AP-014\", \"AP-015\"]\nSEED = 42\n\nMLX_CACHE = Path.home() / \".cache\" / \"mlx-4bit\"\n\nOPENROUTER_URL = \"https://openrouter.ai/api/v1/chat/completions\"\nOPENROUTER_JUDGE = \"google/\" + judge.JUDGE_MODEL\n\n# Qwen3 templates can open a reasoning block. It is not part of the scenario,\n# so it is dropped before scoring.\nTHINK_RE = re.compile(r\"<think>.*?</think>\\s*\", re.DOTALL)\n\n\ndef job_key(model_key: str, path_id: str, prompt_id: str) -> str:\n    return f\"{model_key}|{path_id}|{prompt_id}\"\n\n\ndef read_jsonl(path: Path) -> list:\n    if not path.exists():\n        return []\n    with path.open(encoding=\"utf-8\") as f:\n        return [json.loads(line) for line in f if line.strip()]\n\n\ndef append_jsonl(path: Path, row: dict) -> None:\n    with path.open(\"a\", encoding=\"utf-8\") as f:\n        f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef build_record(prompt_entry: dict, attack_path: dict) -> dict:\n    \"\"\"The same instruction and input that prompt_generator.py sends.\"\"\"\n    return {\n        \"instruction\": pg.SYSTEM_INSTRUCTION,\n        \"input\": pg.build_input_block(prompt_entry, attack_path),\n    }\n\n\ndef render_prompt(model_key: str, record: dict, tokenizer) -> str:\n    \"\"\"Same messages for both models, each in its own chat template.\"\"\"\n    if model_key == MISTRAL:\n        # The exact prompt the adapter was trained on.\n        return prompt_format.build_chat(record, tokenizer, for_training=False)\n\n    return tokenizer.apply_chat_template(\n        prompt_format.build_messages(record, for_training=False),\n        tokenize=False,\n        add_generation_prompt=True,\n        enable_thinking=False,\n    )\n\n\ndef load_model(model_key: str, args):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    model_id = args.mistral_base if model_key == MISTRAL else args.redsage_model\n    print(f\"[*] Loading {LABELS[model_key]} ({model_id}) in 4-bit\")\n\n    model = AutoModelForCausalLM.from_pretrained(\n        model_id,\n        quantization_config=pg.build_bnb_config(),\n        device_map=\"auto\",\n    )\n    tokenizer = AutoTokenizer.from_pretrained(model_id)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    if model_key == MISTRAL:\n        from peft import PeftModel\n        print(f\"[*] Attaching LoRA adapter from {args.adapter_path}\")\n        model = PeftModel.from_pretrained(model, args.adapter_path)\n\n    model.eval()\n    return model, tokenizer\n\n\ndef generate(model, tokenizer, prompt: str) -> dict:\n    \"\"\"Same decoding settings as prompt_generator.py, seeded per call.\"\"\"\n    import torch\n\n    torch.manual_seed(SEED)\n    inputs = tokenizer(\n        prompt,\n        return_tensors=\"pt\",\n        truncation=True,\n        max_length=pg.MAX_INPUT_TOKENS,\n        add_special_tokens=False,\n    ).to(model.device)\n    n = inputs[\"input_ids\"].shape[1]\n\n    started = time.time()\n    with torch.no_grad():\n        output_ids = model.generate(\n            **inputs,\n            max_new_tokens=pg.MAX_NEW_TOKENS,\n            do_sample=True,\n            temperature=0.7,\n            top_p=0.9,\n            repetition_penalty=1.05,\n            pad_token_id=tokenizer.pad_token_id,\n        )\n    seconds = time.time() - started\n\n    text = tokenizer.decode(output_ids[0][n:], skip_special_tokens=True)\n    return {\n        \"output\": THINK_RE.sub(\"\", text).strip(),\n        \"prompt_tokens\": n,\n        \"new_tokens\": int(output_ids.shape[1] - n),\n        \"seconds\": round(seconds, 1),\n    }\n\n\ndef mlx_quantized(hf_id: str) -> str:\n    \"\"\"A 4-bit MLX copy of a Hugging Face model, converted on first use and cached.\"\"\"\n    path = MLX_CACHE / hf_id.replace(\"/\", \"--\")\n    if not path.exists():\n        from mlx_lm import convert\n        print(f\"[*] Converting {hf_id} to 4-bit MLX in {path} (first run only)\")\n        convert(hf_id, mlx_path=str(path), quantize=True, q_bits=4, q_group_size=64)\n    return str(path)\n\n\ndef mlx_adapter(peft_dir: str, out_dir: Path) -> str:\n    \"\"\"The PEFT LoRA adapter in MLX's layout.\n\n    PEFT stores A as (r, in) and B as (out, r) and scales by alpha / r. MLX\n    stores the transposes and takes the scale from its config.\n    \"\"\"\n    import mlx.core as mx\n\n    config = json.loads((Path(peft_dir) / \"adapter_config.json\").read_text())\n    weights = mx.load(str(Path(peft_dir) / \"adapter_model.safetensors\"))\n\n    converted = {}\n    for name, value in weights.items():\n        # base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight\n        module, kind = name[len(\"base_model.model.\"):].rsplit(\".lora_\", 1)\n        converted[f\"{module}.lora_{kind[0].lower()}\"] = value.T\n\n    modules = {tuple(name.rsplit(\".lora_\", 1)[0].split(\".\")) for name in converted}\n    out_dir.mkdir(parents=True, exist_ok=True)\n    mx.save_safetensors(str(out_dir / \"adapters.safetensors\"), converted)\n    (out_dir / \"adapter_config.json\").write_text(json.dumps({\n        \"fine_tune_type\": \"lora\",\n        \"num_layers\": 1 + max(int(parts[2]) for parts in modules),\n        \"lora_parameters\": {\n            \"rank\": config[\"r\"],\n            \"scale\": config[\"lora_alpha\"] / config[\"r\"],\n            \"dropout\": 0.0,\n            \"keys\": sorted({\".\".join(parts[3:]) for parts in modules}),\n        },\n    }, indent=2))\n    return str(out_dir)\n\n\ndef load_model_mlx(model_key: str, args):\n    from mlx_lm import load\n\n    if model_key == MISTRAL:\n        print(f\"[*] Loading {LABELS[MISTRAL]} with the adapter from {args.adapter_path}\")\n        adapter = mlx_adapter(args.adapter_path, Path(args.out_dir) / \"mlx_adapter\")\n        return load(mlx_quantized(args.mistral_base), adapter_path=adapter)\n\n    print(f\"[*] Loading {LABELS[REDSAGE]}\")\n    return load(mlx_quantized(args.redsage_model))\n\n\ndef generate_mlx(model, tokenizer, prompt: str) -> dict:\n    \"\"\"Same decoding settings as the CUDA path.\"\"\"\n    import mlx.core as mx\n    from mlx_lm import stream_generate\n    from mlx_lm.sample_utils import make_logits_processors, make_sampler\n\n    mx.random.seed(SEED)\n    ids = tokenizer.encode(prompt, add_special_tokens=False)\n\n    started = time.time()\n    text, last = \"\", None\n    for last in stream_generate(\n        model, tokenizer, ids,\n        max_tokens=pg.MAX_NEW_TOKENS,\n        sampler=make_sampler(temp=0.7, top_p=0.9),\n        logits_processors=make_logits_processors(repetition_penalty=1.05),\n    ):\n        text += last.text\n\n    return {\n        \"output\": THINK_RE.sub(\"\", text).strip(),\n        \"prompt_tokens\": len(ids),\n        \"new_tokens\": last.generation_tokens,\n        \"seconds\": round(time.time() - started, 1),\n    }\n\n\ndef free_mlx(model, tokenizer) -> None:\n    import gc\n    import mlx.core as mx\n\n    del model, tokenizer\n    gc.collect()\n    mx.clear_cache()\n\n\nBACKENDS = {\n    \"cuda\": (load_model, generate, pg.cleanup_memory),\n    \"mlx\": (load_model_mlx, generate_mlx, free_mlx),\n}\n\n\ndef run_model(model_key: str, jobs: list, args, outputs_path: Path, done: set) -> None:\n    pending = [(p, q) for p, q in jobs\n               if job_key(model_key, p[\"source_id\"], q[\"prompt_id\"]) not in done]\n    if not pending:\n        print(f\"[=] {LABELS[model_key]}: all {len(jobs)} generations already on disk\")\n        return\n\n    load, generate, free = BACKENDS[args.backend]\n    model, tokenizer = load(model_key, args)\n    model_id = args.mistral_base if model_key == MISTRAL else args.redsage_model\n\n    for i, (attack_path, prompt_entry) in enumerate(pending, 1):\n        print(f\"[{LABELS[model_key]} {i}/{len(pending)}] \"\n              f\"{attack_path['source_id']} x {prompt_entry['prompt_id']}\")\n\n        prompt = render_prompt(model_key, build_record(prompt_entry, attack_path), tokenizer)\n        result = generate(model, tokenizer, prompt)\n        print(f\"    {result['new_tokens']} tokens in {result['seconds']}s\")\n\n        append_jsonl(outputs_path, {\n            \"model\": model_key,\n            \"model_id\": model_id,\n            \"path_id\": attack_path[\"source_id\"],\n            \"path_name\": attack_path[\"source_name\"],\n            \"attack_level\": attack_path[\"attack_level\"],\n            \"techniques\": attack_path[\"techniques\"],\n            \"prompt_id\": prompt_entry[\"prompt_id\"],\n            \"focus\": prompt_entry[\"focus\"],\n            **result,\n        })\n\n    free(model, tokenizer)\n\n\ndef openrouter_reply(api_key: str, prompt: str) -> str:\n    \"\"\"One reply from the judge model, retried on rate limits and server errors.\"\"\"\n    body = json.dumps({\n        \"model\": OPENROUTER_JUDGE,\n        \"messages\": [{\"role\": \"user\", \"content\": prompt}],\n    }).encode(\"utf-8\")\n    headers = {\"Authorization\": f\"Bearer {api_key}\", \"Content-Type\": \"application/json\"}\n    last = judge.GEMINI_MAX_RETRIES - 1\n\n    for attempt in range(judge.GEMINI_MAX_RETRIES):\n        request = urllib.request.Request(OPENROUTER_URL, data=body, headers=headers)\n        try:\n            with urllib.request.urlopen(request, timeout=120) as response:\n                return json.load(response)[\"choices\"][0][\"message\"][\"content\"]\n        except urllib.error.HTTPError as e:\n            if (e.code != 429 and e.code < 500) or attempt == last:\n                raise\n        except urllib.error.URLError:\n            if attempt == last:\n                raise\n        time.sleep(min(judge.GEMINI_RETRY_BASE_SECONDS * 2 ** attempt,\n                       judge.GEMINI_RETRY_MAX_SECONDS))\n\n\ndef build_scorer(args):\n    \"\"\"The judge as a function of (scenario, label), checked before any model loads.\"\"\"\n    if args.no_judge:\n        return None\n\n    openrouter_key = os.getenv(\"OPENROUTER_API_KEY\")\n    if openrouter_key:\n        print(f\"[*] Judge: {OPENROUTER_JUDGE} through OpenRouter\")\n        try:\n            openrouter_reply(openrouter_key, \"Reply with exactly: ok\")\n        except urllib.error.HTTPError as e:\n            sys.exit(f\"[!] OpenRouter refused the judge check ({e.code}): \"\n                     f\"{e.read().decode('utf-8', 'replace')[:200]}\")\n\n        def score(text: str, label: str) -> dict:\n            try:\n                reply = openrouter_reply(openrouter_key, judge.build_rubric_prompt(text))\n            except Exception as e:\n                return judge.failed_scores(f\"judge call failed for {label}: {e}\")\n            return judge.parse_score_json(reply)\n\n        return score\n\n    api_key = judge.resolve_api_key(args.gemini_key)\n    if api_key:\n        client = judge.build_judge_client(api_key)\n        judge.preflight_judge_auth(client)\n        return lambda text, label: judge.judge_scenario_safe(client, text, label)\n\n    print(\"[!] No judge key found, so only the format metrics will be reported. \"\n          \"Set OPENROUTER_API_KEY or GOOGLE_API_KEY to add the judge.\")\n    return None\n\n\ndef judge_all(rows: list, scores_path: Path, score) -> dict:\n    \"\"\"Judge scores per generation, keyed like the outputs file.\"\"\"\n    scores = {r[\"key\"]: r[\"scores\"] for r in read_jsonl(scores_path)}\n\n    for row in rows:\n        key = job_key(row[\"model\"], row[\"path_id\"], row[\"prompt_id\"])\n        if key in scores and judge.is_scored(scores[key]):\n            continue\n\n        result = score(row[\"output\"], key)\n        scores[key] = result\n        append_jsonl(scores_path, {\"key\": key, \"scores\": result})\n        time.sleep(judge.GEMINI_CALL_DELAY)\n\n    return scores\n\n\ndef average(values: list) -> float:\n    return round(sum(values) / len(values), 4) if values else 0.0\n\n\ndef winner(name: str, a: float, b: float) -> str:\n    if a == b:\n        return \"Tie\"\n    a_wins = a > b if name not in metrics.LOWER_IS_BETTER else a < b\n    return LABELS[MISTRAL] if a_wins else LABELS[REDSAGE]\n\n\ndef write_report(rows: list, scores: dict, args, out_dir: Path) -> None:\n    by_model = {MISTRAL: [], REDSAGE: []}\n    for row in rows:\n        by_model[row[\"model\"]].append(row)\n\n    measured = {\n        key: [metrics.compute_metrics(r) for r in group]\n        for key, group in by_model.items()\n    }\n\n    lines = [\n        \"# Fine-tuned Mistral vs RedSage Qwen\",\n        \"\",\n        f\"- Fine-tuned Mistral: `{args.mistral_base}` + adapter `{args.adapter_path}`\",\n        f\"- RedSage Qwen: `{args.redsage_model}`\",\n        f\"- Attack paths: `{args.attack_paths}`, \"\n        f\"{len({r['path_id'] for r in rows})} paths, prompts \"\n        f\"{sorted({r['prompt_id'] for r in rows})}\",\n        f\"- Both in 4-bit ({'MLX, group size 64' if args.backend == 'mlx' else 'bitsandbytes NF4'}), \"\n        f\"temperature 0.7, top_p 0.9, repetition penalty 1.05, max {pg.MAX_NEW_TOKENS} new tokens, seed {SEED}\",\n        \"\",\n        \"## Format and coverage metrics (averaged)\",\n        \"\",\n        f\"| Metric | {LABELS[MISTRAL]} | {LABELS[REDSAGE]} | Better |\",\n        \"|---|---:|---:|:---:|\",\n    ]\n    for name in metrics.NUMERIC_METRICS:\n        a = average([getattr(m, name) for m in measured[MISTRAL]])\n        b = average([getattr(m, name) for m in measured[REDSAGE]])\n        lines.append(f\"| `{name}` | {a} | {b} | {winner(name, a, b)} |\")\n\n    speed = {\n        key: average([r[\"new_tokens\"] / r[\"seconds\"] for r in group if r[\"seconds\"]])\n        for key, group in by_model.items()\n    }\n    lines += [\n        \"\",\n        f\"Generation speed: {LABELS[MISTRAL]} {speed[MISTRAL]} tok/s, \"\n        f\"{LABELS[REDSAGE]} {speed[REDSAGE]} tok/s.\",\n    ]\n\n    def judged(model_key, path_id, prompt_id):\n        result = scores.get(job_key(model_key, path_id, prompt_id))\n        return result if result and judge.is_scored(result) else None\n\n    if scores:\n        lines += [\"\", \"## Gemini judge (1 to 10 per criterion)\", \"\",\n                  f\"| Criterion | {LABELS[MISTRAL]} | {LABELS[REDSAGE]} |\",\n                  \"|---|---:|---:|\"]\n        scored = {\n            key: [s for s in (judged(key, r[\"path_id\"], r[\"prompt_id\"]) for r in group) if s]\n            for key, group in by_model.items()\n        }\n        for criterion in judge.SCORE_KEYS + (\"total\",):\n            a = average([s[criterion] for s in scored[MISTRAL]])\n            b = average([s[criterion] for s in scored[REDSAGE]])\n            lines.append(f\"| {criterion} | {a} | {b} |\")\n\n        wins = {MISTRAL: 0, REDSAGE: 0, \"tie\": 0}\n        lines += [\"\", \"| Path | Prompt | Mistral total | RedSage total |\",\n                  \"|---|---|---:|---:|\"]\n        for row in by_model[MISTRAL]:\n            a = judged(MISTRAL, row[\"path_id\"], row[\"prompt_id\"])\n            b = judged(REDSAGE, row[\"path_id\"], row[\"prompt_id\"])\n            if not (a and b):\n                continue\n            wins[MISTRAL if a[\"total\"] > b[\"total\"] else REDSAGE if b[\"total\"] > a[\"total\"] else \"tie\"] += 1\n            lines.append(f\"| {row['path_id']} | {row['prompt_id']} | {a['total']} | {b['total']} |\")\n        lines += [\"\", f\"Entries won: {LABELS[MISTRAL]} {wins[MISTRAL]}, \"\n                      f\"{LABELS[REDSAGE]} {wins[REDSAGE]}, ties {wins['tie']}.\"]\n\n    report = out_dir / \"report.md\"\n    report.write_text(\"\\n\".join(lines) + \"\\n\", encoding=\"utf-8\")\n    print(\"\\n\".join(lines))\n    print(f\"\\n[OK] Report saved to {report}\")\n\n\ndef write_side_by_side(rows: list, out_dir: Path) -> None:\n    \"\"\"Both outputs for each path, one after the other, for reading.\"\"\"\n    outputs = {job_key(r[\"model\"], r[\"path_id\"], r[\"prompt_id\"]): r for r in rows}\n    rule = \"=\" * 80\n    blocks = []\n    for row in rows:\n        if row[\"model\"] != MISTRAL:\n            continue\n        other = outputs.get(job_key(REDSAGE, row[\"path_id\"], row[\"prompt_id\"]))\n        blocks.append(\n            f\"{rule}\\n{row['path_id']} | {row['attack_level']} | {row['prompt_id']}: {row['focus']}\\n\"\n            f\"{row['path_name']}\\nTechniques: {row['techniques']}\\n{rule}\\n\\n\"\n            f\"--- {LABELS[MISTRAL].upper()} ---\\n\\n{row['output']}\\n\\n\"\n            f\"--- {LABELS[REDSAGE].upper()} ---\\n\\n{other['output'] if other else '(not generated)'}\\n\"\n        )\n    path = out_dir / \"side_by_side.txt\"\n    path.write_text(\"\\n\".join(blocks), encoding=\"utf-8\")\n    print(f\"[OK] Outputs saved to {path}\")\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=\"Fine-tuned Mistral-7B vs RedSage-Qwen3-8B on the same attack paths\")\n    parser.add_argument(\"--attack_paths\", default=str(ATTACK_PATHS_DEFAULT),\n                        help=\"attack_paths.json exported by the infrastructure repo\")\n    parser.add_argument(\"--adapter_path\", default=pg.ADAPTER_PATH_DEFAULT,\n                        help=\"LoRA adapter folder or Hugging Face repo id\")\n    parser.add_argument(\"--mistral_base\", default=pg.BASE_MODEL_DEFAULT)\n    parser.add_argument(\"--redsage_model\", default=REDSAGE_DEFAULT)\n    parser.add_argument(\"--prompt_ids\", nargs=\"+\", default=[\"P1\"],\n                        help=\"Prompts from prompt_generator.py to run (default: P1)\")\n    parser.add_argument(\"--path_ids\", nargs=\"+\", default=SAMPLE_PATH_IDS,\n                        help=f\"Paths to run (default: {' '.join(SAMPLE_PATH_IDS)})\")\n    parser.add_argument(\"--all_paths\", action=\"store_true\",\n                        help=\"Run every path in the file instead of --path_ids\")\n    parser.add_argument(\"--out_dir\", default=\"results/model_comparison\")\n    parser.add_argument(\"--backend\", choices=sorted(BACKENDS),\n                        default=\"mlx\" if sys.platform == \"darwin\" else \"cuda\",\n                        help=\"cuda: bitsandbytes on an NVIDIA GPU. mlx: Apple Silicon\")\n    parser.add_argument(\"--gemini_key\", default=None,\n                        help=\"Gemini API key; falls back to GOOGLE_API_KEY / GEMINI_API_KEY\")\n    parser.add_argument(\"--no_judge\", action=\"store_true\",\n                        help=\"Skip the judge and report format metrics only\")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n\n    attack_paths = pg.load_attack_paths(Path(args.attack_paths))\n    if not args.all_paths:\n        attack_paths = [p for p in attack_paths if p[\"source_id\"] in set(args.path_ids)]\n    prompts = [p for p in pg.STRUCTURED_PROMPTS if p[\"prompt_id\"] in set(args.prompt_ids)]\n    if not attack_paths or not prompts:\n        sys.exit(\"[!] No paths or prompts selected. Check --path_ids and --prompt_ids.\")\n\n    jobs = [(p, q) for p in attack_paths for q in prompts]\n    print(f\"[*] {len(attack_paths)} paths x {len(prompts)} prompts x 2 models \"\n          f\"= {2 * len(jobs)} generations\")\n\n    # Check the judge before spending half an hour on generation.\n    score = build_scorer(args)\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    outputs_path = out_dir / \"outputs.jsonl\"\n\n    for model_key in (MISTRAL, REDSAGE):\n        done = {job_key(r[\"model\"], r[\"path_id\"], r[\"prompt_id\"]) for r in read_jsonl(outputs_path)}\n        run_model(model_key, jobs, args, outputs_path, done)\n\n    wanted = {(p[\"source_id\"], q[\"prompt_id\"]) for p, q in jobs}\n    rows = [r for r in read_jsonl(outputs_path) if (r[\"path_id\"], r[\"prompt_id\"]) in wanted]\n\n    scores = judge_all(rows, out_dir / \"judge_scores.jsonl\", score) if score else {}\n\n    write_side_by_side(rows, out_dir)\n    write_report(rows, scores, args, out_dir)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("prompt_generator.py").write_text("\"\"\"prompt_generator.py\"\"\"\n\nimport argparse\nimport csv\nimport gc\nimport json\nimport sys\nfrom pathlib import Path\n\n# Shared canonical prompt builder (the same one training used) — keeps the\nsys.path.insert(0, str(Path(__file__).resolve().parent / \"fine_tuning\"))\nimport prompt_format  # noqa: E402\n\n# ─────────────────────────────────────────────────────────────────────────────\n\nBASE_MODEL_DEFAULT  = \"mistralai/Mistral-7B-Instruct-v0.3\"\nADAPTER_PATH_DEFAULT = \"fine_tuning/output\"\nMAX_NEW_TOKENS       = 1200\nMAX_INPUT_TOKENS     = 2048          # matches finetuning.MAX_SEQ_LEN\n\n\n# ─────────────────────────────────────────────────────────────────────────────\n\nSYSTEM_INSTRUCTION = (\n    \"You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.\\n\"\n    \"Your task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.\\n\\n\"\n    \"CRITICAL FORMATTING RULES — follow these exactly or the output will be rejected:\\n\"\n    \"1. Your response MUST start immediately with the line: [NATURAL LANGUAGE]\\n\"\n    \"2. Do NOT use markdown. No **, no ##, no *, no backticks, no bold, no headers.\\n\"\n    \"3. The three section headers must appear EXACTLY as plain text on their own line:\\n\"\n    \"   [NATURAL LANGUAGE]\\n\"\n    \"   [MATHEMATICAL EXPRESSION]\\n\"\n    \"   [ATTACK GRAPH]\\n\"\n    \"4. Never skip a section. Never add a preamble before [NATURAL LANGUAGE]. Never add closing remarks after [ATTACK GRAPH].\\n\"\n    \"5. Use real MITRE ATT&CK technique IDs exactly as provided.\\n\"\n    \"6. The mathematical expression must follow G = (N, S, V, P, FS) notation exactly.\\n\"\n    \"7. The attack graph must list numbered stages with plain text arrows (->).\"\n)\n\n\n# ─────────────────────────────────────────────────────────────────────────────\n\nSTRUCTURED_PROMPTS = [\n    {\n        \"prompt_id\": \"P1\",\n        \"focus\": \"Full Tri-Modal Scenario (Standard)\",\n        \"user_context\": (\n            \"Generate a complete tri-modal cyber attack scenario for the attack path above.\\n\"\n            \"Produce all three sections in order:\\n\"\n            \"  [NATURAL LANGUAGE]  — A step-by-step narrative: initial access, lateral movement, \"\n            \"privilege escalation, data exfiltration/impact. Reference each MITRE technique ID inline.\\n\"\n            \"  [MATHEMATICAL EXPRESSION] — Graph notation: N (nodes), S (security weaknesses), \"\n            \"V (techniques), P (paths), FS (final state), using G = (N, S, V, P, FS).\\n\"\n            \"  [ATTACK GRAPH] — Numbered stages with plain-text arrows (->).\"\n        ),\n    },\n    {\n        \"prompt_id\": \"P2\",\n        \"focus\": \"Cloud-Centric IAM Abuse Focus\",\n        \"user_context\": (\n            \"Generate a tri-modal cyber attack scenario with a deep focus on cloud IAM misconfigurations.\\n\"\n            \"In [NATURAL LANGUAGE]: emphasise how the attacker abuses IAM roles, service accounts, \"\n            \"metadata APIs, and cloud-native privilege escalation to move from initial access to full \"\n            \"cloud environment compromise. Reference each MITRE technique ID inline.\\n\"\n            \"In [MATHEMATICAL EXPRESSION]: model IAM roles as explicit nodes and misconfigurations as \"\n            \"security weaknesses in the G = (N, S, V, P, FS) notation.\\n\"\n            \"In [ATTACK GRAPH]: trace each stage from endpoint to cloud control plane to final impact.\"\n        ),\n    },\n    {\n        \"prompt_id\": \"P3\",\n        \"focus\": \"Stealth & Persistence — APT-Style\",\n        \"user_context\": (\n            \"Generate a tri-modal cyber attack scenario modelled on a long-dwell APT campaign.\\n\"\n            \"In [NATURAL LANGUAGE]: emphasise how the attacker maintains stealth over weeks — \"\n            \"slow credential harvesting, living-off-the-land techniques, log tampering, and \"\n            \"establishing persistent backdoors — before achieving final objectives. \"\n            \"Reference each MITRE technique ID inline.\\n\"\n            \"In [MATHEMATICAL EXPRESSION]: highlight persistence nodes and defense-evasion \"\n            \"weaknesses within the G = (N, S, V, P, FS) notation.\\n\"\n            \"In [ATTACK GRAPH]: show the full kill chain from initial compromise through \"\n            \"persistence, discovery, lateral movement, and final impact.\"\n        ),\n    },\n    {\n        \"prompt_id\": \"P4\",\n        \"focus\": \"Impact-Oriented — Ransomware / Data Exfiltration\",\n        \"user_context\": (\n            \"Generate a tri-modal cyber attack scenario focused on the impact phase: \"\n            \"large-scale data exfiltration or ransomware deployment against cloud-hosted assets.\\n\"\n            \"In [NATURAL LANGUAGE]: describe initial access and lateral movement concisely, \"\n            \"then provide rich detail on the data staging, encryption, exfiltration mechanism, \"\n            \"and business impact (regulatory fines, reputational damage, service disruption). \"\n            \"Reference each MITRE technique ID inline.\\n\"\n            \"In [MATHEMATICAL EXPRESSION]: include egress nodes and final-state nodes \"\n            \"in the G = (N, S, V, P, FS) notation.\\n\"\n            \"In [ATTACK GRAPH]: ensure the final stage explicitly names the impact \"\n            \"(e.g., Ransomware Deployment, Data Breach, Service Disruption).\"\n        ),\n    },\n]\n\n\n# ─────────────────────────────────────────────────────────────────────────────\n\nATTACK_PATH_KEYS = (\n    \"source_id\",\n    \"source_name\",\n    \"industry\",\n    \"attack_level\",\n    \"techniques\",\n    \"surfaces\",\n\n    # Added with the exporter's objectives layer. The same hops are a different\n    # scenario depending on what the attacker does at the end — reading a\n    # database, encrypting it, or flooding it — and this is the only field that\n    # says which. Without it the model would write a data-theft narrative for a\n    # ransomware path.\n    \"objective\",\n)\n\n\ndef load_attack_paths(path: Path) -> list:\n    \"\"\"Load ranked attack paths from the exported JSON list, worst first.\"\"\"\n    if not path.exists():\n        raise FileNotFoundError(\n            f\"attack paths file not found: {path}\\n\"\n            \"    Generate it in the infrastructure repo with:\\n\"\n            \"        python3 -m runner.graph_runner\\n\"\n            \"    then pass its output/attack_paths.json via --attack_paths.\"\n        )\n\n    with path.open(\"r\", encoding=\"utf-8\") as f:\n        data = json.load(f)\n\n    if not isinstance(data, list):\n        raise ValueError(\n            f\"{path}: expected a JSON list of attack paths, got \"\n            f\"{type(data).__name__}\"\n        )\n\n    if not data:\n        raise ValueError(\n            f\"{path}: the export contains no attack paths. The graph found \"\n            \"nothing exploitable — check the findings and edges upstream \"\n            \"before generating scenarios.\"\n        )\n\n    for i, entry in enumerate(data):\n        if not isinstance(entry, dict):\n            raise ValueError(f\"{path}: entry {i} is not a JSON object\")\n        missing = [key for key in ATTACK_PATH_KEYS if key not in entry]\n        if missing:\n            raise ValueError(\n                f\"{path}: entry {i} is missing required key(s) \"\n                f\"{', '.join(missing)} — the prompt template reads all of \"\n                f\"{', '.join(ATTACK_PATH_KEYS)}. Re-run the exporter.\"\n            )\n\n    return data\n\n\n# ─────────────────────────────────────────────────────────────────────────────\n\ndef build_bnb_config():\n    \"\"\"4-bit QLoRA quantisation — same config used during fine-tuning.\"\"\"\n    import torch\n    from transformers import BitsAndBytesConfig\n\n    return BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n\n\ndef build_input_block(prompt_entry: dict, attack_path: dict) -> str:\n    \"\"\"Build the record's ``input`` block — the facts half of the prompt.\"\"\"\n    return (\n        f\"SOURCE: {attack_path['source_name']} ({attack_path['source_id']})\\n\"\n        f\"INDUSTRY: {attack_path['industry']}\\n\"\n        f\"ATTACK LEVEL: {attack_path['attack_level']}\\n\"\n        f\"TECHNIQUES: {attack_path['techniques']}\\n\"\n        f\"SURFACES: {attack_path['surfaces']}\\n\"\n        f\"OBJECTIVE: {attack_path['objective']}\\n\\n\"\n        f\"PROMPT FOCUS: {prompt_entry['focus']}\\n\"\n        f\"{prompt_entry['user_context']}\"\n    )\n\n\ndef format_model_input(prompt_entry: dict, attack_path: dict, tokenizer=None) -> str:\n    \"\"\"Return the generation prompt for one (prompt, attack path) pair — the\"\"\"\n    record = {\n        \"instruction\": SYSTEM_INSTRUCTION,\n        \"input\": build_input_block(prompt_entry, attack_path),\n    }\n    return prompt_format.build_chat(record, tokenizer, for_training=False)\n\n\ndef generate_text(model, tokenizer, prompt_entry: dict, attack_path: dict) -> str:\n    \"\"\"Generate one scenario and return only the newly generated text.\"\"\"\n    import torch\n\n    prompt = format_model_input(prompt_entry, attack_path, tokenizer)\n    inputs = tokenizer(\n        prompt,\n        return_tensors=\"pt\",\n        truncation=True,\n        max_length=MAX_INPUT_TOKENS,\n        add_special_tokens=False,\n    ).to(model.device)\n    n = inputs[\"input_ids\"].shape[1]\n\n    with torch.no_grad():\n        output_ids = model.generate(\n            **inputs,\n            max_new_tokens=MAX_NEW_TOKENS,\n            do_sample=True,\n            temperature=0.7,\n            top_p=0.9,\n            repetition_penalty=1.05,\n            eos_token_id=tokenizer.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id,\n        )\n\n    return tokenizer.decode(output_ids[0][n:], skip_special_tokens=True).strip()\n\n\ndef cleanup_memory(model=None, tokenizer=None):\n    import torch\n\n    if model is not None:\n        del model\n    if tokenizer is not None:\n        del tokenizer\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef save_results_csv(results: list[dict], path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fieldnames = [\"prompt_id\", \"focus\", \"attack_path_id\", \"industry\",\n                  \"attack_level\", \"techniques\", \"surfaces\",\n                  \"full_prompt\", \"generated_scenario\"]\n    with path.open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        for r in results:\n            writer.writerow(r)\n    print(f\"[✓] CSV saved  → {path}\")\n\n\ndef save_results_txt(results: list[dict], path: Path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    sep = \"=\" * 80\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        f.write(\"PROMPT GENERATOR — SCENARIO GENERATION OUTPUT\\n\")\n        f.write(f\"{sep}\\n\\n\")\n        for r in results:\n            f.write(f\"PROMPT ID    : {r['prompt_id']}\\n\")\n            f.write(f\"FOCUS        : {r['focus']}\\n\")\n            f.write(f\"ATTACK PATH  : {r['attack_path_id']} | {r['industry']} | LEVEL: {r['attack_level']}\\n\")\n            f.write(f\"TECHNIQUES   : {r['techniques']}\\n\")\n            f.write(f\"SURFACES     : {r['surfaces']}\\n\")\n            f.write(f\"\\n--- GENERATED SCENARIO ---\\n\\n\")\n            f.write(r[\"generated_scenario\"])\n            f.write(f\"\\n\\n{sep}\\n\\n\")\n    print(f\"[✓] TXT saved  → {path}\")\n\n\n# ─────────────────────────────────────────────────────────────────────────────\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=\"Prompt Generator — tri-modal scenario generation over ranked attack paths via fine-tuned Mistral-7B\"\n    )\n    parser.add_argument(\n        \"--base_model\",\n        default=BASE_MODEL_DEFAULT,\n        help=f\"Base HuggingFace model name or local path (default: {BASE_MODEL_DEFAULT})\",\n    )\n    parser.add_argument(\n        \"--adapter_path\",\n        default=ADAPTER_PATH_DEFAULT,\n        help=f\"Path to LoRA adapter directory (default: {ADAPTER_PATH_DEFAULT})\",\n    )\n    parser.add_argument(\n        \"--output_prefix\",\n        default=\"results/prompt_gen_output\",\n        help=\"Output file prefix — produces <prefix>.csv and <prefix>.txt (default: results/prompt_gen_output)\",\n    )\n    parser.add_argument(\n        \"--attack_paths\",\n        required=True,\n        help=\"Path to the ranked attack paths exported by the infrastructure repo's \"\n             \"runner/graph_runner.py (output/attack_paths.json) — a JSON list whose entries \"\n             \"carry source_id, source_name, industry, attack_level, techniques, surfaces. \"\n             \"Every prompt is run against every path in the file\",\n    )\n    parser.add_argument(\n        \"--prompt_ids\",\n        nargs=\"+\",\n        default=None,\n        help=\"Run only specific prompt IDs, e.g. --prompt_ids P1 P3  (default: all)\",\n    )\n    parser.add_argument(\n        \"--path_ids\",\n        nargs=\"+\",\n        default=None,\n        help=\"Run only specific attack-path source IDs, e.g. --path_ids AP-001 AP-002 \"\n             \"(default: all paths in the file)\",\n    )\n    return parser.parse_args()\n\n\n# ─────────────────────────────────────────────────────────────────────────────\n\ndef main() -> None:\n    args = parse_args()\n\n    adapter_path  = Path(args.adapter_path)\n    attack_paths_path = Path(args.attack_paths)\n    output_csv    = Path(f\"{args.output_prefix}.csv\")\n    output_txt    = Path(f\"{args.output_prefix}.txt\")\n\n    # ── Attack paths, before anything expensive happens ───────────────────────\n    try:\n        attack_paths = load_attack_paths(attack_paths_path)\n    except (FileNotFoundError, ValueError, json.JSONDecodeError) as e:\n        print(f\"[!] Could not load attack paths: {e}\")\n        sys.exit(1)\n\n    if args.path_ids:\n        wanted = set(args.path_ids)\n        attack_paths = [p for p in attack_paths if p[\"source_id\"] in wanted]\n        if not attack_paths:\n            print(f\"[!] None of the requested path IDs {args.path_ids} matched \"\n                  f\"{attack_paths_path}.\")\n            sys.exit(1)\n\n    # Filter prompt list if --prompt_ids was specified\n    prompts_to_run = STRUCTURED_PROMPTS\n    if args.prompt_ids:\n        ids = set(args.prompt_ids)\n        prompts_to_run = [p for p in STRUCTURED_PROMPTS if p[\"prompt_id\"] in ids]\n        if not prompts_to_run:\n            print(f\"[!] None of the requested prompt IDs {args.prompt_ids} matched. \"\n                  f\"Available: {[p['prompt_id'] for p in STRUCTURED_PROMPTS]}\")\n            sys.exit(1)\n\n    print()\n    print(\"=\" * 70)\n    print(\"  PROMPT GENERATOR — Intelligent Cloud Security Testing\")\n    print(\"  Synthetic Cyber Attack Scenario Generation\")\n    print(\"=\" * 70)\n    print(f\"  Base model   : {args.base_model}\")\n    print(f\"  Adapter path : {adapter_path}\")\n    print(f\"  Attack paths : {attack_paths_path} ({len(attack_paths)} paths)\")\n    print(f\"  Output CSV   : {output_csv}\")\n    print(f\"  Output TXT   : {output_txt}\")\n    print(f\"  Prompts      : {[p['prompt_id'] for p in prompts_to_run]}\")\n    print(f\"  Paths        : {[p['source_id'] for p in attack_paths]}\")\n    print(f\"  Generations  : {len(prompts_to_run) * len(attack_paths)}\")\n    print(\"=\" * 70)\n    print()\n\n    if not adapter_path.exists():\n        print(f\"[!] Adapter path not found: {adapter_path}\")\n        print(\"    Run this script from the scenario_generation/ directory.\")\n        sys.exit(1)\n\n    # ── Load model ────────────────────────────────────────────────────────────\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    if not torch.cuda.is_available():\n        print(\"[!] WARNING: No GPU detected. Generation will be very slow on CPU.\")\n\n    print(\"[*] Loading base model in 4-bit (QLoRA)…\")\n    bnb_config = build_bnb_config()\n\n    try:\n        base_model = AutoModelForCausalLM.from_pretrained(\n            args.base_model,\n            quantization_config=bnb_config,\n            device_map=\"auto\",\n        )\n    except Exception as e:\n        print(f\"[!] Failed to load base model: {e}\")\n        sys.exit(1)\n\n    tokenizer = AutoTokenizer.from_pretrained(args.base_model)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = \"right\"\n\n    print(f\"[*] Loading LoRA adapter from: {adapter_path}\")\n    try:\n        finetuned_model = PeftModel.from_pretrained(base_model, str(adapter_path))\n    except Exception as e:\n        print(f\"[!] Failed to load LoRA adapter: {e}\")\n        cleanup_memory(base_model, tokenizer)\n        sys.exit(1)\n    finetuned_model.eval()\n    print(\"[✓] Model ready.\\n\")\n\n    # ── Generation loop ───────────────────────────────────────────────────────\n    results = []\n    total = len(prompts_to_run) * len(attack_paths)\n    i = 0\n\n    for attack_path in attack_paths:\n        for prompt_entry in prompts_to_run:\n            i += 1\n            pid    = prompt_entry[\"prompt_id\"]\n            focus  = prompt_entry[\"focus\"]\n\n            print(f\"{'─'*70}\")\n            print(f\"[{i}/{total}] Path {attack_path['source_id']} \"\n                  f\"({attack_path['attack_level']}) × Prompt {pid}: {focus}\")\n            print(f\"    {attack_path['source_name']}\")\n            print(f\"{'─'*70}\")\n\n            # Recorded for the CSV. generate_text() rebuilds it through the same\n            full_prompt = format_model_input(prompt_entry, attack_path, tokenizer)\n\n            print(\"[*] Generating scenario…\")\n            scenario = generate_text(\n                finetuned_model, tokenizer, prompt_entry, attack_path)\n\n            # ── Terminal preview ──────────────────────────────────────────────\n            print(\"\\n┌── GENERATED SCENARIO (preview first 600 chars) ──────────────┐\")\n            preview = scenario[:600] + (\"\\n  …[truncated]\" if len(scenario) > 600 else \"\")\n            for line in preview.splitlines():\n                print(f\"│  {line}\")\n            print(\"└──────────────────────────────────────────────────────────────┘\\n\")\n\n            results.append({\n                \"prompt_id\"          : pid,\n                \"focus\"              : focus,\n                \"attack_path_id\"     : attack_path[\"source_id\"],\n                \"industry\"           : attack_path[\"industry\"],\n                \"attack_level\"       : attack_path[\"attack_level\"],\n                \"techniques\"         : attack_path[\"techniques\"],\n                \"surfaces\"           : attack_path[\"surfaces\"],\n                \"full_prompt\"        : full_prompt,\n                \"generated_scenario\" : scenario,\n            })\n\n    # ── Cleanup ───────────────────────────────────────────────────────────────\n    print(\"[*] Unloading model and clearing GPU cache…\")\n    cleanup_memory(finetuned_model, tokenizer)\n\n    # ── Save outputs ──────────────────────────────────────────────────────────\n    print(\"\\n=== SAVING RESULTS ===\")\n    save_results_csv(results, output_csv)\n    save_results_txt(results, output_txt)\n\n    print(\"\\n[✓] Done. Review the outputs in the files above.\")\n    print(\"    Compare these against baseline Mistral-7B using evaluation/evaluate.py\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("prompt_format.py").write_text("\"\"\"\nprompt_format.py\n----------------\nTHE single source of truth for the prompt that wraps each training record and\neach inference request. Training (finetuning.py) and every inference path\n(run_inference.py, evaluation/evaluate.py, the generated notebooks) all import\n``build_chat`` from here, so the train-time and infer-time prompts can never\ndrift apart. Prompt drift was the original root cause of the off-format model\noutput, so this module is deliberately small and shared.\n\nWhat the prompt looks like for Mistral-7B-Instruct-v0.3\n------------------------------------------------------\nMistral's chat template has no separate ``system`` role that survives a single\nuser turn, so we fold the full ``instruction`` (the tri-modal formatting rules)\ninto the user turn together with the record's ``input``::\n\n    <s>[INST] {instruction}\\\\n\\\\n{input}[/INST]                 # inference prompt\n    <s>[INST] {instruction}\\\\n\\\\n{input}[/INST] {output}</s>    # full training text\n\nThe inference prompt is, by construction, a strict prefix of the training text,\nwhich is exactly what the model needs to continue the completion at inference.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\n\n# Mistral-7B-Instruct-v0.3 special tokens. Used only by the no-dependency manual\n# fallback below; when a real tokenizer is supplied we read these from it.\nDEFAULT_BOS = \"<s>\"\nDEFAULT_EOS = \"</s>\"\n\n# The marker that separates the prompt from the completion. Loss is computed only\n# on the tokens that follow this marker (see finetuning.py completion-only setup).\nRESPONSE_MARKER = \"[/INST]\"\n\n\ndef build_messages(record: dict, for_training: bool) -> list[dict]:\n    \"\"\"\n    Build the chat ``messages`` list for one record.\n\n    The full ``instruction`` field (all the [NATURAL LANGUAGE] / [MATHEMATICAL\n    EXPRESSION] / [ATTACK GRAPH] rules) is folded into the user turn — never the\n    short one-liner that older code used.\n    \"\"\"\n    instruction = (record.get(\"instruction\") or \"\").rstrip()\n    input_text = record.get(\"input\") or \"\"\n    user_content = f\"{instruction}\\n\\n{input_text}\"\n\n    messages = [{\"role\": \"user\", \"content\": user_content}]\n    if for_training:\n        output = (record.get(\"output\") or \"\").strip()\n        messages.append({\"role\": \"assistant\", \"content\": output})\n    return messages\n\n\ndef _manual_chat(record: dict, for_training: bool,\n                 bos: str = DEFAULT_BOS, eos: str = DEFAULT_EOS) -> str:\n    \"\"\"\n    Dependency-free fallback that reproduces Mistral's chat template byte-for-byte.\n\n    Mirrors output/chat_template.jinja:\n      user turn      -> \"[INST] \" + content + \"[/INST]\"   (bos prepended once)\n      assistant turn -> \" \" + content.strip() + eos\n    Used when no tokenizer is available (e.g. ``--check_data`` on a machine with\n    no transformers installed). The infer string stays a prefix of the train\n    string here too.\n    \"\"\"\n    instruction = (record.get(\"instruction\") or \"\").rstrip()\n    input_text = record.get(\"input\") or \"\"\n    prompt = f\"{bos}[INST] {instruction}\\n\\n{input_text}[/INST]\"\n    if not for_training:\n        return prompt\n    output = (record.get(\"output\") or \"\").strip()\n    return f\"{prompt} {output}{eos}\"\n\n\ndef build_chat(record: dict, tokenizer: Optional[object], for_training: bool) -> str:\n    \"\"\"\n    Return the canonical prompt string for ``record``.\n\n    Parameters\n    ----------\n    record       : dict with at least ``instruction`` and ``input`` (and\n                   ``output`` when ``for_training`` is True).\n    tokenizer    : a HF tokenizer (uses ``apply_chat_template``) or ``None``\n                   (falls back to the manual template above).\n    for_training : True  -> include the assistant turn + EOS (full training text)\n                   False -> stop after ``[/INST]`` (the generation prompt)\n\n    Guarantee: ``build_chat(rec, tok, False)`` is always a strict prefix of\n    ``build_chat(rec, tok, True)`` for the same record/tokenizer.\n    \"\"\"\n    if tokenizer is not None and hasattr(tokenizer, \"apply_chat_template\"):\n        try:\n            return tokenizer.apply_chat_template(\n                build_messages(record, for_training),\n                tokenize=False,\n                add_generation_prompt=not for_training,\n            )\n        except Exception:\n            # Some odd template/version combo — fall back to the manual template,\n            # using the tokenizer's real special tokens so it stays consistent.\n            bos = getattr(tokenizer, \"bos_token\", None) or DEFAULT_BOS\n            eos = getattr(tokenizer, \"eos_token\", None) or DEFAULT_EOS\n            return _manual_chat(record, for_training, bos, eos)\n    return _manual_chat(record, for_training)\n\n\ndef assert_parity(records: list[dict], tokenizer: Optional[object] = None) -> None:\n    \"\"\"\n    Assert that the inference prompt is a strict prefix of the full training text\n    for every record. This is the train/inference-parity guard (fix A acceptance).\n    Raises AssertionError on the first violation.\n    \"\"\"\n    for i, rec in enumerate(records):\n        infer = build_chat(rec, tokenizer, for_training=False)\n        train = build_chat(rec, tokenizer, for_training=True)\n        assert train.startswith(infer), (\n            f\"Parity violated for record {i} \"\n            f\"(source_id={rec.get('_meta', {}).get('source_id')}): \"\n            \"inference prompt is not a prefix of the training text.\"\n        )\n        assert len(train) > len(infer), (\n            f\"Record {i} produced an empty completion span.\"\n        )\n\n\ndef response_template_ids(tokenizer, sample_text: str, marker: str = RESPONSE_MARKER) -> list[int]:\n    \"\"\"\n    Compute the token ids of ``marker`` (``[/INST]``) **in context**.\n\n    The same string can tokenize differently standalone vs. mid-sequence, which\n    is the classic footgun that makes DataCollatorForCompletionOnlyLM silently\n    mask the entire example. We therefore diff the tokenization of the text up to\n    *and including* the marker against the text up to *just before* it, and use\n    the trailing ids — the ids the marker actually contributes in this context.\n\n    Requires a real tokenizer; only called on the training/smoke path.\n    \"\"\"\n    cut = sample_text.index(marker)\n    end = cut + len(marker)\n    before_ids = tokenizer(sample_text[:cut], add_special_tokens=False)[\"input_ids\"]\n    upto_ids = tokenizer(sample_text[:end], add_special_tokens=False)[\"input_ids\"]\n\n    # The marker's in-context ids are whatever was appended after `before_ids`.\n    # Guard against rare boundary re-tokenization of the last shared token.\n    k = 0\n    while k < len(before_ids) and k < len(upto_ids) and before_ids[k] == upto_ids[k]:\n        k += 1\n    ids = upto_ids[k:]\n    if not ids:\n        # Fallback: standalone encoding (better than an empty template).\n        ids = tokenizer(marker, add_special_tokens=False)[\"input_ids\"]\n    return ids\n", encoding='utf-8')
pathlib.Path("data_split.py").write_text("\"\"\"\ndata_split.py\n-------------\nTHE single source of truth for splitting the dataset into train / validation /\ntest. Persisting the split here (and reading it everywhere) removes the old\ntrain/eval mismatch where finetuning.py did its own random split while\nmake_test_inputs.py treated a different file's tail as \"unseen\" — so the\n\"unseen\" test set was not guaranteed unseen.\n\nHow it works\n------------\n* The split is computed once, deterministically, from a single seed, keyed on\n  each record's stable ``_meta.source_id``.\n* The held-out **test** records are written to ``evaluation/test_inputs.jsonl``.\n* finetuning.py trains on the **complement** of that file (train + val only);\n  make_test_inputs.py creates it; evaluate.py reads it. Everyone shares one\n  artifact, so train ∩ test is provably empty.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport random\nfrom pathlib import Path\nfrom typing import Optional\n\n\nDEFAULT_TEST_INPUTS = \"evaluation/test_inputs.jsonl\"\n\n\ndef load_records(jsonl_path: str | Path) -> list[dict]:\n    \"\"\"Read a JSONL dataset, skipping blank / malformed lines.\"\"\"\n    records = []\n    with open(jsonl_path, \"r\", encoding=\"utf-8\") as f:\n        for line in f:\n            line = line.strip()\n            if not line:\n                continue\n            try:\n                records.append(json.loads(line))\n            except json.JSONDecodeError:\n                continue\n    return records\n\n\ndef source_id(record: dict, idx: int) -> str:\n    \"\"\"Stable id for a record: its _meta.source_id, else a positional fallback.\"\"\"\n    sid = (record.get(\"_meta\") or {}).get(\"source_id\")\n    return sid if sid else f\"idx-{idx}\"\n\n\ndef _annotate(records: list[dict]) -> list[dict]:\n    \"\"\"Attach the original index + canonical source_id to each record (copy).\"\"\"\n    out = []\n    for i, rec in enumerate(records):\n        item = dict(rec)\n        item[\"idx\"] = i\n        item[\"source_id\"] = source_id(rec, i)\n        out.append(item)\n    return out\n\n\ndef compute_split(records: list[dict], test_size: int = 10,\n                  val_frac: float = 0.15, seed: int = 42) -> dict:\n    \"\"\"\n    Deterministically split annotated records into test / val / train.\n\n    test_size : number of held-out test records (default 10).\n    val_frac  : fraction of the *remaining* pool used for validation\n                (always at least 1 record so eval_loss can be computed).\n    Returns {\"train\": [...], \"val\": [...], \"test\": [...]} with records sorted by\n    original index within each split.\n    \"\"\"\n    annotated = _annotate(records)\n    n = len(annotated)\n    if test_size >= n:\n        raise ValueError(f\"test_size={test_size} >= dataset size {n}\")\n\n    order = list(range(n))\n    random.Random(seed).shuffle(order)\n\n    test_idx = order[:test_size]\n    pool_idx = order[test_size:]\n    val_count = max(1, round(val_frac * len(pool_idx)))\n    val_count = min(val_count, len(pool_idx) - 1)  # keep at least 1 for train\n    val_idx = pool_idx[:val_count]\n    train_idx = pool_idx[val_count:]\n\n    by_idx = {item[\"idx\"]: item for item in annotated}\n    pick = lambda idxs: [by_idx[i] for i in sorted(idxs)]\n    return {\"train\": pick(train_idx), \"val\": pick(val_idx), \"test\": pick(test_idx)}\n\n\n# ── Persisted test-set artifact ────────────────────────────────────────────────\n\n# Fields we keep in test_inputs.jsonl. `output` is the reference completion (for\n# eval comparison only — never fed to the model as input).\nTEST_FIELDS = (\"idx\", \"source_id\", \"instruction\", \"input\", \"output\")\n\n\ndef write_test_inputs(path: str | Path, test_records: list[dict]) -> None:\n    \"\"\"Persist the held-out test records to a JSONL artifact.\"\"\"\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", encoding=\"utf-8\") as f:\n        for rec in test_records:\n            row = {k: rec.get(k, \"\") for k in TEST_FIELDS}\n            f.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n\n\ndef read_test_inputs(path: str | Path) -> list[dict]:\n    \"\"\"Read the held-out test artifact.\"\"\"\n    return load_records(path)\n\n\ndef ids_of(records: list[dict]) -> set:\n    \"\"\"Set of canonical ids (idx, source_id) tuples for intersection checks.\"\"\"\n    out = set()\n    for i, rec in enumerate(records):\n        out.add((rec.get(\"idx\", i), rec.get(\"source_id\") or source_id(rec, i)))\n    return out\n\n\ndef ensure_test_inputs(data_path: str | Path,\n                       test_inputs_path: str | Path = DEFAULT_TEST_INPUTS,\n                       test_size: int = 10, val_frac: float = 0.15,\n                       seed: int = 42) -> dict:\n    \"\"\"\n    Return the split for ``data_path``, creating ``test_inputs_path`` from the\n    deterministic split if it does not exist yet. When it already exists, the\n    test split is taken from the persisted artifact (source of truth) and the\n    train/val pool is the complement, so all callers agree.\n    \"\"\"\n    records = load_records(data_path)\n    split = compute_split(records, test_size=test_size, val_frac=val_frac, seed=seed)\n\n    test_path = Path(test_inputs_path)\n    if not test_path.exists():\n        write_test_inputs(test_path, split[\"test\"])\n        return split\n\n    # Artifact exists. If it matches the seed-derived split exactly, use the\n    # canonical split so train/val are identical regardless of who created the\n    # file first (finetuning vs make_test_inputs).\n    persisted_ids = {row.get(\"source_id\") for row in read_test_inputs(test_path)}\n    if persisted_ids == {r[\"source_id\"] for r in split[\"test\"]}:\n        return split\n\n    # A custom / hand-edited artifact: respect it and derive the complement.\n    print(f\"[i] {test_path} differs from the seed-derived split; using the persisted test set.\")\n    annotated = _annotate(records)\n    pool = [r for r in annotated if r[\"source_id\"] not in persisted_ids]\n    test = [r for r in annotated if r[\"source_id\"] in persisted_ids]\n    order = list(range(len(pool)))\n    random.Random(seed).shuffle(order)\n    val_count = max(1, min(round(val_frac * len(pool)), len(pool) - 1))\n    val_idx = set(order[:val_count])\n    val = [pool[i] for i in sorted(val_idx)]\n    train = [pool[i] for i in range(len(pool)) if i not in val_idx]\n    return {\"train\": train, \"val\": val, \"test\": test}\n", encoding='utf-8')
pathlib.Path("judge_eval.py").write_text("\"\"\"judge_eval.py\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport sys\nimport time\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Optional\n\n# Shared canonical modules from fine_tuning/ (dependency-free — they import fine\n# with no ML stack installed). Same import shim as evaluate.py / make_test_inputs.py.\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"fine_tuning\"))\nimport prompt_format  # noqa: E402\nimport data_split  # noqa: E402\n\n\n# ── Constants ─────────────────────────────────────────────────────────────────\n\nJUDGE_MODEL = \"gemini-2.5-flash-lite\"\n\nMAX_NEW_TOKENS = 1280        # matches evaluate.py / run_inference.py\nMAX_INPUT_TOKENS = 2048      # matches finetuning.MAX_SEQ_LEN\n\nGEMINI_CALL_DELAY = 3        # seconds between judge calls, to stay off the 429 path\nGEMINI_MAX_RETRIES = 6\nGEMINI_RETRY_BASE_SECONDS = 5\nGEMINI_RETRY_MAX_SECONDS = 60\n\n# The rubric criteria, in report order. \"total\" is always the computed sum.\nSCORE_KEYS = (\"level_of_detail\", \"technical_accuracy\", \"realism\", \"creativity\")\n\n\n# ── Logging helpers — makes a 20-minute run readable ──────────────────────────\n\ndef _ts() -> str:\n    \"\"\"Return a short timestamp like [08:42:01] for every log line.\"\"\"\n    return datetime.now().strftime(\"[%H:%M:%S]\")\n\n\ndef log(msg: str) -> None:\n    \"\"\"Standard info message.\"\"\"\n    print(f\"{_ts()} [INFO]  {msg}\", flush=True)\n\n\ndef log_ok(msg: str) -> None:\n    \"\"\"Success message.\"\"\"\n    print(f\"{_ts()} [ OK ]  {msg}\", flush=True)\n\n\ndef log_warn(msg: str) -> None:\n    \"\"\"Warning — something is off but we can continue.\"\"\"\n    print(f\"{_ts()} [WARN]  {msg}\", flush=True)\n\n\ndef log_err(msg: str) -> None:\n    \"\"\"Error — something failed.\"\"\"\n    print(f\"{_ts()} [ERR!]  {msg}\", flush=True)\n\n\ndef log_section(title: str) -> None:\n    \"\"\"Print a big separator so sections are findable in long output.\"\"\"\n    bar = \"=\" * 60\n    print(f\"\\n{bar}\", flush=True)\n    print(f\"  {title}\", flush=True)\n    print(f\"{bar}\", flush=True)\n\n\ndef log_output_preview(label: str, text: str, max_chars: int = 300) -> None:\n    \"\"\"Print the first max_chars characters of a generated output, so you can see at\"\"\"\n    preview = text[:max_chars].replace(\"\\n\", \" \").strip()\n    if len(text) > max_chars:\n        preview += f\"... [{len(text)} chars total]\"\n    print(f\"{_ts()} [TEXT]  {label}: {preview}\", flush=True)\n\n\ndef log_scores(label: str, scores: dict) -> None:\n    \"\"\"Pretty-print a score dict so each criterion is on its own line.\"\"\"\n    print(f\"{_ts()} [SCORE] {label}:\", flush=True)\n    print(f\"           detail={scores['level_of_detail']:>2}  \"\n          f\"accuracy={scores['technical_accuracy']:>2}  \"\n          f\"realism={scores['realism']:>2}  \"\n          f\"creativity={scores['creativity']:>2}  \"\n          f\"-> TOTAL={scores['total']:>2}\", flush=True)\n\n\n# ── API key helpers ───────────────────────────────────────────────────────────\n\ndef normalize_api_key(value) -> Optional[str]:\n    \"\"\"Strip quotes/whitespace; treat an unexpanded {PLACEHOLDER} as missing.\"\"\"\n    if value is None:\n        return None\n    cleaned = value.strip().strip('\"').strip(\"'\")\n    if not cleaned:\n        return None\n    if re.fullmatch(r\"\\{[A-Za-z_][A-Za-z0-9_]*\\}\", cleaned):\n        return None\n    return cleaned\n\n\ndef resolve_api_key(cli_key) -> Optional[str]:\n    \"\"\"CLI flag wins, then GOOGLE_API_KEY, then GEMINI_API_KEY.\"\"\"\n    key = normalize_api_key(cli_key)\n    if key:\n        log(\"API key source: --gemini_key argument\")\n        return key\n    key = normalize_api_key(os.getenv(\"GOOGLE_API_KEY\"))\n    if key:\n        log(\"API key source: GOOGLE_API_KEY environment variable\")\n        return key\n    key = normalize_api_key(os.getenv(\"GEMINI_API_KEY\"))\n    if key:\n        log(\"API key source: GEMINI_API_KEY environment variable\")\n        return key\n    return None\n\n\ndef mask_api_key(key: str) -> str:\n    \"\"\"Never log a whole key — the logs get pasted into the report.\"\"\"\n    if len(key) <= 8:\n        return \"*\" * len(key)\n    return f\"{key[:4]}...{key[-4:]}\"\n\n\n# ── Gemini client + auth ──────────────────────────────────────────────────────\n\ndef build_judge_client(api_key: str):\n    \"\"\"Build the Gemini client from an explicit key.\"\"\"\n    from google import genai\n\n    saved_google = os.environ.pop(\"GOOGLE_API_KEY\", None)\n    saved_gemini = os.environ.pop(\"GEMINI_API_KEY\", None)\n    try:\n        return genai.Client(api_key=api_key)\n    finally:\n        if saved_google is not None:\n            os.environ[\"GOOGLE_API_KEY\"] = saved_google\n        if saved_gemini is not None:\n            os.environ[\"GEMINI_API_KEY\"] = saved_gemini\n\n\ndef is_retryable_gemini_error(exc: Exception) -> bool:\n    \"\"\"True for transient rate-limit / unavailable errors worth backing off on.\"\"\"\n    msg = str(exc).upper()\n    retry_tokens = [\n        \"503\",\n        \"UNAVAILABLE\",\n        \"429\",\n        \"RESOURCE_EXHAUSTED\",\n        \"RATE LIMIT\",\n        \"TOO MANY REQUESTS\",\n    ]\n    return any(token in msg for token in retry_tokens)\n\n\ndef gemini_generate_with_retry(client, model: str, contents: str, call_label: str):\n    \"\"\"Call Gemini with exponential backoff; returns (response, elapsed_seconds).\"\"\"\n    last_error = None\n    for attempt in range(1, GEMINI_MAX_RETRIES + 1):\n        try:\n            t0 = time.time()\n            response = client.models.generate_content(model=model, contents=contents)\n            return response, time.time() - t0\n        except Exception as e:\n            last_error = e\n            if not is_retryable_gemini_error(e) or attempt == GEMINI_MAX_RETRIES:\n                raise\n\n            wait_s = min(GEMINI_RETRY_BASE_SECONDS * (2 ** (attempt - 1)),\n                         GEMINI_RETRY_MAX_SECONDS)\n            log_warn(\n                f\"  [gemini] {call_label} failed (attempt {attempt}/{GEMINI_MAX_RETRIES}) \"\n                f\"with retryable error: {e}. Waiting {wait_s}s then retrying...\"\n            )\n            time.sleep(wait_s)\n\n    # Not reachable (the loop either returns or raises), but keeps analysers happy.\n    if last_error is not None:\n        raise last_error\n\n\ndef preflight_judge_auth(client) -> None:\n    \"\"\"Ping the judge model before loading 7B of weights.\"\"\"\n    log(f\"Sending auth precheck ping to model: {JUDGE_MODEL}\")\n    try:\n        response, elapsed = gemini_generate_with_retry(\n            client=client,\n            model=JUDGE_MODEL,\n            contents=\"Reply with exactly: ok\",\n            call_label=\"auth precheck\",\n        )\n        if not response or not getattr(response, \"text\", None):\n            raise ValueError(\"Empty response during Gemini auth precheck\")\n        log_ok(f\"Auth precheck passed in {elapsed:.1f}s — model replied: \"\n               f\"'{response.text.strip()}'\")\n    except Exception as e:\n        msg = str(e)\n        if \"API_KEY_INVALID\" in msg or \"API Key not found\" in msg:\n            raise RuntimeError(\n                \"Gemini API key was rejected. Make sure GOOGLE_API_KEY is a valid \"\n                \"AI Studio key, not a restricted Cloud key.\"\n            ) from e\n        if \"404\" in msg or \"not found\" in msg.lower():\n            raise RuntimeError(\n                f\"Model '{JUDGE_MODEL}' not found. Check the exact name with \"\n                f\"client.models.list(). Original error: {msg}\"\n            ) from e\n        raise\n\n\n# ── Held-out test set (resolved through the ONE shared split) ─────────────────\n\ndef resolve_test_records(data_path, test_inputs_path, test_size: int = 10,\n                         val_frac: float = 0.15, seed: int = 42) -> list:\n    \"\"\"Return the held-out records to judge.\"\"\"\n    data_path = Path(data_path)\n    test_path = Path(test_inputs_path)\n\n    if data_path.exists():\n        split = data_split.ensure_test_inputs(\n            data_path, test_path,\n            test_size=test_size, val_frac=val_frac, seed=seed,\n        )\n        test_ids = data_split.ids_of(split[\"test\"])\n        pool_ids = data_split.ids_of(split[\"train\"]) | data_split.ids_of(split[\"val\"])\n        if not test_ids.isdisjoint(pool_ids):\n            raise ValueError(\n                \"Held-out test set overlaps the train/val pool — refusing to \"\n                \"judge the fine-tune on data it was trained on.\"\n            )\n        return split[\"test\"]\n\n    if test_path.exists():\n        records = data_split.read_test_inputs(test_path)\n        if not records:\n            raise ValueError(f\"No valid test input entries found in {test_path}\")\n        return records\n\n    raise FileNotFoundError(\n        f\"Neither the dataset ({data_path}) nor the held-out artifact \"\n        f\"({test_path}) exists. Run: python evaluation/make_test_inputs.py\"\n    )\n\n\ndef record_source_id(record: dict, idx: int) -> str:\n    \"\"\"Stable label for one record, agreeing with data_split.source_id.\"\"\"\n    return (record.get(\"source_id\")\n            or (record.get(\"_meta\") or {}).get(\"source_id\")\n            or f\"entry_{idx:02d}\")\n\n\n# ── Prompt + local model generation ───────────────────────────────────────────\n\ndef build_eval_prompt(record: dict, tokenizer) -> str:\n    \"\"\"Return the generation prompt for ``record`` — the canonical one, byte for byte.\"\"\"\n    return prompt_format.build_chat(record, tokenizer, for_training=False)\n\n\ndef build_bnb_config():\n    \"\"\"4-bit NF4 quantization config — identical to evaluate.py's.\"\"\"\n    import torch\n    from transformers import BitsAndBytesConfig\n\n    return BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n\n\ndef load_judged_models(base_model_name: str, adapter_path):\n    \"\"\"Load the 4-bit base model once and attach the LoRA adapter to it.\"\"\"\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from peft import PeftModel\n\n    log(f\"Loading {base_model_name} in 4-bit quantization (NF4)...\")\n    t0 = time.time()\n    model = AutoModelForCausalLM.from_pretrained(\n        base_model_name,\n        quantization_config=build_bnb_config(),\n        device_map=\"auto\",\n    )\n    log_ok(f\"Base model loaded in {time.time() - t0:.1f}s\")\n    log(f\"Device map: {getattr(model, 'hf_device_map', 'N/A')}\")\n\n    tokenizer = AutoTokenizer.from_pretrained(base_model_name)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        log_warn(\"pad_token was None — set to eos_token\")\n    tokenizer.padding_side = \"right\"\n    log_ok(\"Tokenizer loaded\")\n\n    log(f\"Attaching LoRA adapter from: {adapter_path}\")\n    t0 = time.time()\n    model = PeftModel.from_pretrained(model, str(adapter_path))\n    model.eval()\n    log_ok(f\"Adapter attached in {time.time() - t0:.1f}s\")\n    return model, tokenizer\n\n\ndef generate_text(model, tokenizer, record: dict, label: str = \"model\") -> str:\n    \"\"\"Run the local model on one record and return only the newly generated text.\"\"\"\n    import torch\n\n    prompt = build_eval_prompt(record, tokenizer)\n    inputs = tokenizer(prompt, return_tensors=\"pt\", truncation=True,\n                       max_length=MAX_INPUT_TOKENS,\n                       add_special_tokens=False).to(model.device)\n    n = inputs[\"input_ids\"].shape[1]\n    log(f\"  [{label}] Generating... input tokens={n}, max_new_tokens={MAX_NEW_TOKENS}\")\n\n    t0 = time.time()\n    with torch.no_grad():\n        output_ids = model.generate(\n            **inputs,\n            max_new_tokens=MAX_NEW_TOKENS,\n            do_sample=True,\n            temperature=0.7,\n            top_p=0.9,\n            repetition_penalty=1.05,\n            eos_token_id=tokenizer.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id,\n        )\n    elapsed = time.time() - t0\n\n    new_tokens = output_ids.shape[1] - n\n    tok_per_sec = new_tokens / elapsed if elapsed > 0 else 0\n    log_ok(f\"  [{label}] Done — {new_tokens} new tokens in {elapsed:.1f}s \"\n           f\"({tok_per_sec:.1f} tok/s)\")\n\n    return tokenizer.decode(output_ids[0][n:], skip_special_tokens=True).strip()\n\n\ndef generate_pair(model, tokenizer, record: dict) -> tuple:\n    \"\"\"Return (base_output, finetuned_output) for one record from the single model.\"\"\"\n    log(\"  Generating from BASE model (adapter disabled)...\")\n    with model.disable_adapter():\n        base_output = generate_text(model, tokenizer, record, label=\"base\")\n    log_output_preview(\"BASE output\", base_output)\n\n    log(\"  Generating from FINE-TUNED model...\")\n    tuned_output = generate_text(model, tokenizer, record, label=\"ft\")\n    log_output_preview(\"FINE-TUNED output\", tuned_output)\n    return base_output, tuned_output\n\n\n# ── Gemini judge + score parsing ──────────────────────────────────────────────\n\ndef extract_json_block(text: str) -> Optional[str]:\n    \"\"\"Pull the JSON object out of a judge reply, or return None.\"\"\"\n    text = (text or \"\").strip()\n    if text.startswith(\"```\"):\n        text = re.sub(r\"^```(?:json)?\\s*\", \"\", text)\n        text = re.sub(r\"\\s*```$\", \"\", text)\n        text = text.strip()\n    if text.startswith(\"{\") and text.endswith(\"}\"):\n        return text\n    match = re.search(r\"\\{.*\\}\", text, flags=re.DOTALL)\n    if match:\n        return match.group(0)\n    return None\n\n\ndef failed_scores(reason: str) -> dict:\n    \"\"\"A zeroed score dict carrying why it failed.\"\"\"\n    scores = {key: 0 for key in SCORE_KEYS}\n    scores[\"total\"] = 0\n    scores[\"parse_error\"] = reason\n    return scores\n\n\ndef is_scored(scores: dict) -> bool:\n    \"\"\"True when this dict holds real judge scores (not a parse failure).\"\"\"\n    return \"parse_error\" not in scores\n\n\ndef parse_score_json(text: str) -> dict:\n    \"\"\"Parse a judge reply into {criterion: int, ..., \"total\": int}.\"\"\"\n    raw_json = extract_json_block(text)\n    if raw_json is None:\n        return failed_scores(f\"no JSON object in judge reply: {(text or '')[:120]!r}\")\n\n    try:\n        data = json.loads(raw_json.replace(\"'\", '\"'))\n    except json.JSONDecodeError as e:\n        return failed_scores(f\"judge reply is not valid JSON ({e})\")\n    if not isinstance(data, dict):\n        return failed_scores(f\"judge reply JSON is a {type(data).__name__}, not an object\")\n\n    scores = {}\n    for key in SCORE_KEYS:\n        if key not in data:\n            return failed_scores(f\"missing key {key!r} in score JSON\")\n        try:\n            scores[key] = int(data[key])\n        except (TypeError, ValueError):\n            return failed_scores(f\"non-numeric value for {key!r}: {data[key]!r}\")\n\n    computed_total = sum(scores[key] for key in SCORE_KEYS)\n    try:\n        reported_total = int(data.get(\"total\", computed_total))\n    except (TypeError, ValueError):\n        reported_total = computed_total\n    if reported_total != computed_total:\n        log_warn(f\"Judge total mismatch: reported={reported_total}, \"\n                 f\"computed={computed_total}. Using the computed sum.\")\n    scores[\"total\"] = computed_total\n    return scores\n\n\ndef build_rubric_prompt(scenario_text: str) -> str:\n    \"\"\"The judge prompt: fixed rubric, JSON-only reply, no reference answer.\"\"\"\n    return (\n        \"Score this cyber attack scenario on 4 criteria, each 1-10.\\n\"\n        \"Return ONLY valid JSON with this exact structure, no explanation, no markdown:\\n\"\n        '{\"level_of_detail\": <int 1-10>, '\n        '\"technical_accuracy\": <int 1-10>, '\n        '\"realism\": <int 1-10>, '\n        '\"creativity\": <int 1-10>, '\n        '\"total\": <sum of above>}\\n'\n        f\"Scenario to score:\\n{scenario_text}\"\n    )\n\n\ndef judge_scenario(client, scenario_text: str, label: str = \"output\") -> dict:\n    \"\"\"Ask Gemini to score one scenario. Logs the raw reply so a formatting problem\"\"\"\n    log(f\"  [gemini] Judging: {label} (scenario length: {len(scenario_text)} chars)\")\n\n    response, elapsed = gemini_generate_with_retry(\n        client=client,\n        model=JUDGE_MODEL,\n        contents=build_rubric_prompt(scenario_text),\n        call_label=f\"judge {label}\",\n    )\n\n    if not response or not getattr(response, \"text\", None):\n        return failed_scores(\"empty response from Gemini judge\")\n\n    log_ok(f\"  [gemini] Response received in {elapsed:.1f}s\")\n    raw = response.text.strip()\n    log(f\"  [gemini] Raw response: {raw[:200]}\")\n    return parse_score_json(raw)\n\n\ndef judge_scenario_safe(client, scenario_text: str, label: str) -> dict:\n    \"\"\"judge_scenario, but a transport failure degrades this entry only.\"\"\"\n    try:\n        return judge_scenario(client, scenario_text, label=label)\n    except Exception as e:\n        log_err(f\"  [gemini] judge call failed for {label}: {e}\")\n        return failed_scores(f\"judge call failed: {e}\")\n\n\n# ── Results table ─────────────────────────────────────────────────────────────\n\ndef print_table(per_entry: list, base_avg: float, tuned_avg: float,\n                improvement: float) -> None:\n    \"\"\"Per-entry totals, then the averages and the overall improvement.\"\"\"\n    log_section(\"FINAL RESULTS\")\n    header = (f\"{'source_id':<16}{'base_total':>12}{'finetuned_total':>18}\"\n              f\"{'delta':>10}{'winner':>12}\")\n    print(header)\n    print(\"-\" * len(header))\n    for row in per_entry:\n        sid = row[\"source_id\"]\n        b = row[\"base_scores\"][\"total\"]\n        t = row[\"finetuned_scores\"][\"total\"]\n        d = t - b\n        winner = \"fine-tuned\" if d > 0 else (\"base\" if d < 0 else \"tie\")\n        print(f\"{sid:<16}{b:>12}{t:>18}{d:>10}{winner:>12}\")\n    print(\"-\" * len(header))\n    print(f\"{'AVERAGE':<16}{base_avg:>12.2f}{tuned_avg:>18.2f}\"\n          f\"{(tuned_avg - base_avg):>10.2f}\")\n    improvement_label = f\"+{improvement:.2f}%\" if improvement >= 0 else f\"{improvement:.2f}%\"\n    print(f\"\\n[OK] Overall improvement: {improvement_label}\")\n\n\n# ── Argument parsing ──────────────────────────────────────────────────────────\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=\"Score base vs fine-tuned scenarios with a Gemini judge\")\n    parser.add_argument(\"--base_model\", required=True,\n                        help=\"Base model path or HF name (e.g. mistralai/Mistral-7B-Instruct-v0.3)\")\n    parser.add_argument(\"--adapter_path\", required=True,\n                        help=\"Path to the LoRA adapter output folder (e.g. fine_tuning/output)\")\n    parser.add_argument(\"--test_inputs\", required=True,\n                        help=\"Path to the shared held-out split artifact (evaluation/test_inputs.jsonl)\")\n    parser.add_argument(\"--out\", required=True,\n                        help=\"Path to save the judge scores JSON (e.g. evaluation/judge_scores.json)\")\n    parser.add_argument(\"--gemini_key\", \"--google_api_key\", dest=\"gemini_key\", default=None,\n                        help=\"Gemini API key; falls back to GOOGLE_API_KEY / GEMINI_API_KEY / .env\")\n    parser.add_argument(\"--data\", default=\"dataset/training_data.jsonl\",\n                        help=\"Full dataset JSONL the held-out split is derived from \"\n                             \"(default: dataset/training_data.jsonl)\")\n    parser.add_argument(\"--test_size\", type=int, default=10,\n                        help=\"Number of held-out records to judge (default: 10)\")\n    parser.add_argument(\"--val_frac\", type=float, default=0.15,\n                        help=\"Validation fraction of the train pool, as used in training (default: 0.15)\")\n    parser.add_argument(\"--seed\", type=int, default=42,\n                        help=\"Split seed; must match the training run (default: 42)\")\n    return parser.parse_args()\n\n\n# ── Main ──────────────────────────────────────────────────────────────────────\n\ndef main() -> None:\n    args = parse_args()\n\n    try:\n        log_section(\"STARTUP\")\n        run_start = time.time()\n\n        dotenv_path = Path(__file__).resolve().parents[1] / \".env\"\n        if dotenv_path.exists():\n            try:\n                from dotenv import load_dotenv\n            except ImportError:\n                log_warn(\"python-dotenv not installed — skipping .env \"\n                         \"(pip install -r evaluation/requirements_eval.txt)\")\n            else:\n                load_dotenv(dotenv_path=dotenv_path, override=False)\n                log(f\"Loaded .env from: {dotenv_path}\")\n\n        api_key = resolve_api_key(args.gemini_key)\n        if not api_key:\n            log_err(\"Gemini API key missing. Use --gemini_key or set GOOGLE_API_KEY.\")\n            sys.exit(1)\n\n        log(f\"API key: {mask_api_key(api_key)} (len={len(api_key)})\")\n        log(f\"Judge model: {JUDGE_MODEL}\")\n        log(f\"Gemini call delay: {GEMINI_CALL_DELAY}s between requests\")\n        log(f\"Max new tokens: {MAX_NEW_TOKENS}\")\n        log(f\"Base model: {args.base_model}\")\n        log(f\"Adapter path: {args.adapter_path}\")\n        log(f\"Dataset: {args.data}\")\n        log(f\"Test inputs: {args.test_inputs}\")\n        log(f\"Output path: {args.out}\")\n\n        adapter_path = Path(args.adapter_path)\n        out_path = Path(args.out)\n\n        if not adapter_path.exists():\n            log_err(f\"Adapter path not found: {adapter_path}\")\n            sys.exit(1)\n\n        # ── Resolve the held-out set ────────────────\n        log_section(\"RESOLVING HELD-OUT TEST SET\")\n        entries = resolve_test_records(\n            args.data, args.test_inputs,\n            test_size=args.test_size, val_frac=args.val_frac, seed=args.seed,\n        )\n        log_ok(f\"{len(entries)} held-out entries (seed={args.seed}), \"\n               \"disjoint from the train/val pool\")\n        for i, entry in enumerate(entries, start=1):\n            log(f\"  Entry {i:>2}: source_id={record_source_id(entry, i)!r}, \"\n                f\"instruction_len={len(entry.get('instruction', ''))} chars, \"\n                f\"input_len={len(entry.get('input', ''))} chars\")\n\n        # ── Gemini auth check (before loading 7B) ───\n        log_section(\"GEMINI AUTH CHECK\")\n        judge_client = build_judge_client(api_key)\n        preflight_judge_auth(judge_client)\n\n        # ── Load the model + adapter ────────────────\n        log_section(\"LOADING MODEL + ADAPTER\")\n        model, tokenizer = load_judged_models(args.base_model, adapter_path)\n\n        adapter_files = sorted(p for p in adapter_path.iterdir())\n        log(f\"Adapter directory contents ({len(adapter_files)} items):\")\n        for f in adapter_files:\n            if f.is_file():\n                log(f\"  {f.name:<40} {f.stat().st_size / 1e6:>8.2f} MB\")\n            else:\n                log(f\"  {f.name}/ (dir)\")\n\n        # ── Main evaluation loop ────────────────────\n        log_section(\"EVALUATION LOOP\")\n        log(f\"Processing {len(entries)} entries.\")\n        log(f\"Estimated Gemini calls: {len(entries) * 2} (2 per entry)\")\n\n        per_entry = []\n        unscored = []\n        base_totals = []\n        tuned_totals = []\n\n        for idx, entry in enumerate(entries, start=1):\n            source_id = record_source_id(entry, idx)\n            instruction = entry.get(\"instruction\", \"\")\n            input_text = entry.get(\"input\", \"\")\n\n            print(flush=True)\n            log(f\"Entry {idx}/{len(entries)}: {source_id} {'─' * 35}\")\n\n            if not instruction or not input_text:\n                log_warn(f\"Skipping {source_id}: missing instruction or input field\")\n                unscored.append({\"source_id\": source_id,\n                                 \"reason\": \"missing instruction or input field\"})\n                continue\n\n            t_gen = time.time()\n            base_output, tuned_output = generate_pair(model, tokenizer, entry)\n            log(f\"  Both generations done in {time.time() - t_gen:.1f}s total\")\n\n            log(\"  Asking Gemini to judge BASE output...\")\n            base_scores = judge_scenario_safe(judge_client, base_output,\n                                              label=f\"{source_id}/base\")\n            log_scores(\"BASE scores\", base_scores)\n            log(f\"  [gemini] Waiting {GEMINI_CALL_DELAY}s before next judge call...\")\n            time.sleep(GEMINI_CALL_DELAY)\n\n            log(\"  Asking Gemini to judge FINE-TUNED output...\")\n            tuned_scores = judge_scenario_safe(judge_client, tuned_output,\n                                               label=f\"{source_id}/ft\")\n            log_scores(\"FINE-TUNED scores\", tuned_scores)\n            log(f\"  [gemini] Waiting {GEMINI_CALL_DELAY}s before next entry...\")\n            time.sleep(GEMINI_CALL_DELAY)\n\n            if not is_scored(base_scores) or not is_scored(tuned_scores):\n                reason = base_scores.get(\"parse_error\") or tuned_scores.get(\"parse_error\")\n                log_warn(f\"  {source_id} left out of the averages: {reason}\")\n                unscored.append({\"source_id\": source_id, \"reason\": reason})\n                continue\n\n            delta = tuned_scores[\"total\"] - base_scores[\"total\"]\n            if delta > 0:\n                log_ok(f\"  Fine-tuned WINS this entry by {delta} points\")\n            elif delta < 0:\n                log_warn(f\"  Base model wins this entry by {abs(delta)} points\")\n            else:\n                log(\"  TIE for this entry\")\n\n            base_totals.append(base_scores[\"total\"])\n            tuned_totals.append(tuned_scores[\"total\"])\n            per_entry.append({\n                \"source_id\": source_id,\n                \"base_scores\": base_scores,\n                \"finetuned_scores\": tuned_scores,\n                \"base_output\": base_output,\n                \"finetuned_output\": tuned_output,\n            })\n\n        # ── Final results ───────────────────────────\n        if not per_entry:\n            log_err(\"No entries were successfully evaluated. Check your test inputs.\")\n            sys.exit(1)\n\n        base_avg = sum(base_totals) / len(base_totals)\n        tuned_avg = sum(tuned_totals) / len(tuned_totals)\n        improvement_pct = ((tuned_avg - base_avg) / base_avg * 100.0) if base_avg else 0.0\n\n        result = {\n            \"summary\": {\n                \"judge_model\": JUDGE_MODEL,\n                \"entries_scored\": len(per_entry),\n                \"entries_unscored\": len(unscored),\n                \"base_model_avg\": round(base_avg, 3),\n                \"finetuned_model_avg\": round(tuned_avg, 3),\n                \"improvement_pct\": round(improvement_pct, 3),\n            },\n            \"per_entry\": [\n                {\n                    \"source_id\": item[\"source_id\"],\n                    \"base_scores\": item[\"base_scores\"],\n                    \"finetuned_scores\": item[\"finetuned_scores\"],\n                }\n                for item in per_entry\n            ],\n            \"unscored\": unscored,\n        }\n\n        out_path.parent.mkdir(parents=True, exist_ok=True)\n        with out_path.open(\"w\", encoding=\"utf-8\") as f:\n            json.dump(result, f, indent=2)\n\n        print_table(per_entry, base_avg, tuned_avg, improvement_pct)\n        if unscored:\n            log_warn(f\"{len(unscored)} entries were not scored (see 'unscored' in {out_path})\")\n\n        log_ok(f\"Saved judge results to: {out_path}\")\n        log_ok(f\"Total run time: {(time.time() - run_start) / 60:.1f} minutes\")\n\n    except KeyboardInterrupt:\n        print(\"\\n[!] Evaluation interrupted by user\")\n        sys.exit(130)\n    except Exception as e:\n        log_err(f\"Evaluation failed: {e}\")\n        import traceback\n        traceback.print_exc()\n        sys.exit(1)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("manual_eval_metrics.py").write_text("\"\"\"\nmanual_eval_metrics.py\n----------------------\nReference-free, deterministic evaluation of prompt_gen_comparison.txt.\n\nWHY these metrics (not an LLM judge):\n  - Your fine-tuned model was trained on a specific tri-modal format with\n    explicit rules (section headers, G-notation, MITRE IDs, numbered stages).\n  - A generic LLM judge has no knowledge of those rules and would introduce\n    its own biases (length preference, rewording preference, etc.).\n  - These metrics directly test what the training actually optimised for.\n\nUsage:\n    python evaluation/manual_eval_metrics.py --txt prompt_gen_comparison.txt\n    python evaluation/manual_eval_metrics.py --txt prompt_gen_comparison.txt --csv\n\"\"\"\n\nimport argparse\nimport csv\nimport os\nimport re\nimport sys\nfrom dataclasses import dataclass, field, fields\nfrom pathlib import Path\nfrom typing import Optional\n\n\n# ── Section / format constants ────────────────────────────────────────────────\n\nSECTIONS_ORDERED = [\n    \"[NATURAL LANGUAGE]\",\n    \"[MATHEMATICAL EXPRESSION]\",\n    \"[ATTACK GRAPH]\",\n]\n\n# Regex patterns\nMITRE_ID_RE   = re.compile(r'\\bT\\d{4}(?:\\.\\d{3})?\\b')\nG_NOTATION_RE = re.compile(r'G\\s*=\\s*\\(\\s*N\\s*,\\s*S\\s*,\\s*V\\s*,\\s*P\\s*,\\s*FS\\s*\\)')\nARROW_RE      = re.compile(r'->')\n# Numbered stage: \"1.\" or \"1)\" or \"Stage 1\" at start of line\nSTAGE_RE      = re.compile(r'(?:^|\\n)\\s*(?:Stage\\s*)?\\d+[.):\\s]', re.MULTILINE)\n# Markdown contamination — things the training explicitly forbade\nMD_RE         = re.compile(r'(\\*\\*|^#{1,3} )', re.MULTILINE)\n# G = (...) node lists — each component defined separately (N = {...}, S = {...} etc.)\nCOMPONENT_RE  = re.compile(r'\\b([NSVPF]S?)\\s*=\\s*\\{', re.MULTILINE)\n# Section content extractor\nNL_TEXT_RE    = re.compile(\n    r'\\[NATURAL LANGUAGE\\](.*?)(?=\\[MATHEMATICAL EXPRESSION\\]|\\Z)', re.DOTALL)\nMATH_TEXT_RE  = re.compile(\n    r'\\[MATHEMATICAL EXPRESSION\\](.*?)(?=\\[ATTACK GRAPH\\]|\\Z)', re.DOTALL)\nGRAPH_TEXT_RE = re.compile(\n    r'\\[ATTACK GRAPH\\](.*?)(?=\\Z)', re.DOTALL)\n\n\n# ── Data class ────────────────────────────────────────────────────────────────\n\n@dataclass\nclass OutputMetrics:\n    model:               str = \"\"\n    prompt_id:           str = \"\"\n    focus:               str = \"\"\n\n    # 1. Structural Compliance (0–3 + bool)\n    sections_found:      int   = 0    # how many of the 3 headers appear\n    sections_in_order:   int   = 0    # 1 if all 3 appear in the correct order\n    all_sections_present:int   = 0    # 1 if sections_found == 3\n\n    # 2. MITRE Technique Coverage\n    mitre_input_count:   int   = 0    # IDs in the prompt\n    mitre_output_count:  int   = 0    # unique IDs found in the output\n    mitre_recall:        float = 0.0  # output ∩ input / input\n    mitre_precision:     float = 0.0  # output ∩ input / output (no hallucination)\n\n    # 3. Format Specificity (training-critical rules)\n    g_notation_exact:    int   = 0    # G = (N, S, V, P, FS) present?\n    g_components_defined:int   = 0    # how many of N/S/V/P/FS are separately defined\n    arrow_count:         int   = 0    # number of -> edges in attack graph\n    numbered_stages:     int   = 0    # numbered stage lines found\n    markdown_violations: int   = 0    # ** / # contamination count (lower = better)\n\n    # 4. Lexical Diversity\n    word_count:          int   = 0    # total words in output\n    nl_word_count:       int   = 0    # words in [NATURAL LANGUAGE] section only\n    distinct_1:          float = 0.0  # unique unigrams / total unigrams\n    distinct_2:          float = 0.0  # unique bigrams / total bigrams\n    type_token_ratio:    float = 0.0  # same as distinct_1, aliased for clarity\n\n    # 5. Completeness signals\n    output_truncated:    int   = 0    # 1 if output ends mid-sentence (no terminal punct)\n    nl_has_mitre_inline: int   = 0    # MITRE IDs appear in NL prose (not just graph)\n    graph_has_arrows:    int   = 0    # attack graph section has at least 1 ->\n\n\n# ── Parser: extract outputs from comparison TXT ──────────────────────────────\n\ndef _extract_technique_ids(techniques_str: str) -> set[str]:\n    return set(MITRE_ID_RE.findall(techniques_str))\n\n\ndef parse_comparison_txt(txt_path: Path) -> list[dict]:\n    \"\"\"\n    Return a list of dicts, each with keys:\n        prompt_id, focus, techniques, model, output\n    Parsed from prompt_gen_comparison.txt.\n    \"\"\"\n    text = txt_path.read_text(encoding=\"utf-8\")\n    records = []\n\n    # Split on the wide separator lines\n    blocks = re.split(r'={40,}', text)\n\n    for block in blocks:\n        block = block.strip()\n        if not block:\n            continue\n\n        # Is this a prompt block?\n        pm = re.match(r'PROMPT (P\\d+): (.+?)\\n', block)\n        if not pm:\n            continue\n        prompt_id = pm.group(1)\n        focus     = pm.group(2).strip()\n\n        # Extract techniques line\n        tech_m = re.search(r'Techniques: (.+?)(?:\\n|$)', block)\n        techniques = tech_m.group(1).strip() if tech_m else \"\"\n\n        # Split into baseline / fine-tuned sub-blocks\n        parts = re.split(\n            r'(BASELINE OUTPUT.*?|FINE-TUNED OUTPUT.*?)\\n-{40,}',\n            block, flags=re.DOTALL\n        )\n        # parts alternates: [before, label, content, label, content, ...]\n        i = 0\n        while i < len(parts):\n            part = parts[i].strip()\n            if \"BASELINE OUTPUT\" in part:\n                model = \"baseline\"\n                output = parts[i + 1].strip() if i + 1 < len(parts) else \"\"\n                records.append({\n                    \"prompt_id\": prompt_id, \"focus\": focus,\n                    \"techniques\": techniques, \"model\": model, \"output\": output\n                })\n                i += 2\n            elif \"FINE-TUNED OUTPUT\" in part:\n                model = \"finetuned\"\n                output = parts[i + 1].strip() if i + 1 < len(parts) else \"\"\n                records.append({\n                    \"prompt_id\": prompt_id, \"focus\": focus,\n                    \"techniques\": techniques, \"model\": model, \"output\": output\n                })\n                i += 2\n            else:\n                i += 1\n\n    return records\n\n\n# ── Metric computation ────────────────────────────────────────────────────────\n\ndef compute_metrics(rec: dict) -> OutputMetrics:\n    m   = OutputMetrics()\n    txt = rec[\"output\"]\n    m.model     = rec[\"model\"]\n    m.prompt_id = rec[\"prompt_id\"]\n    m.focus     = rec[\"focus\"]\n\n    # ── 1. Structural Compliance ──────────────────────────────────────────────\n    present_in_order = [s for s in SECTIONS_ORDERED if s in txt]\n    m.sections_found       = len(present_in_order)\n    m.all_sections_present = int(m.sections_found == 3)\n    m.sections_in_order    = int(present_in_order == SECTIONS_ORDERED)\n\n    # ── 2. MITRE Coverage ────────────────────────────────────────────────────\n    input_ids  = _extract_technique_ids(rec[\"techniques\"])\n    output_ids = set(MITRE_ID_RE.findall(txt))\n    m.mitre_input_count  = len(input_ids)\n    m.mitre_output_count = len(output_ids)\n    if input_ids:\n        m.mitre_recall    = round(len(input_ids & output_ids) / len(input_ids), 3)\n    if output_ids:\n        m.mitre_precision = round(len(input_ids & output_ids) / len(output_ids), 3)\n\n    # ── 3. Format Specificity ─────────────────────────────────────────────────\n    m.g_notation_exact     = int(bool(G_NOTATION_RE.search(txt)))\n    m.g_components_defined = len(set(COMPONENT_RE.findall(txt)))\n    m.arrow_count          = len(ARROW_RE.findall(txt))\n    m.numbered_stages      = len(STAGE_RE.findall(txt))\n    m.markdown_violations  = len(MD_RE.findall(txt))\n\n    # ── 4. Lexical Diversity ─────────────────────────────────────────────────\n    tokens = re.findall(r'\\b\\w+\\b', txt.lower())\n    m.word_count = len(tokens)\n    if len(tokens) >= 2:\n        bigrams = [f\"{tokens[i]}_{tokens[i+1]}\" for i in range(len(tokens) - 1)]\n        m.distinct_1      = round(len(set(tokens))  / len(tokens), 4)\n        m.distinct_2      = round(len(set(bigrams)) / len(bigrams), 4)\n        m.type_token_ratio = m.distinct_1\n\n    # Natural-language section word count\n    nl_m = NL_TEXT_RE.search(txt)\n    if nl_m:\n        nl_tokens = re.findall(r'\\b\\w+\\b', nl_m.group(1).lower())\n        m.nl_word_count = len(nl_tokens)\n        # MITRE IDs in NL prose — fine-tuned should reference them inline\n        m.nl_has_mitre_inline = int(bool(MITRE_ID_RE.search(nl_m.group(1))))\n\n    # ── 5. Completeness ──────────────────────────────────────────────────────\n    stripped = txt.rstrip()\n    m.output_truncated = int(\n        bool(stripped) and stripped[-1] not in {'.', '!', '?', '}', ']'}\n    )\n\n    graph_m = GRAPH_TEXT_RE.search(txt)\n    if graph_m:\n        m.graph_has_arrows = int(bool(ARROW_RE.search(graph_m.group(1))))\n\n    return m\n\n\n# ── Reporting ─────────────────────────────────────────────────────────────────\n\nMETRIC_EXPLANATIONS = {\n    \"sections_found\":       \"# of [NL]/[MATH]/[GRAPH] headers found (max 3)\",\n    \"sections_in_order\":    \"All 3 sections appear in the correct order (1=yes)\",\n    \"all_sections_present\": \"All 3 sections present (1=yes)\",\n    \"mitre_recall\":         \"Fraction of INPUT technique IDs that appear in output\",\n    \"mitre_precision\":      \"Fraction of OUTPUT technique IDs that were in the input (hallucination check)\",\n    \"g_notation_exact\":     \"G = (N, S, V, P, FS) exact notation present (1=yes)\",\n    \"g_components_defined\": \"# of G components (N/S/V/P/FS) separately defined (max 5)\",\n    \"arrow_count\":          \"# of -> edges in attack graph (more = richer graph)\",\n    \"numbered_stages\":      \"# of numbered stage lines\",\n    \"markdown_violations\":  \"# of ** / # markdown artefacts (lower=better, should be 0 for fine-tuned)\",\n    \"word_count\":           \"Total word count of output\",\n    \"nl_word_count\":        \"Word count of [NATURAL LANGUAGE] section only\",\n    \"distinct_1\":           \"Lexical diversity: unique unigrams / total unigrams (higher=richer)\",\n    \"distinct_2\":           \"Lexical diversity: unique bigrams / total bigrams (higher=richer)\",\n    \"output_truncated\":     \"Output appears to be cut off mid-sentence (1=truncated, 0=complete)\",\n    \"nl_has_mitre_inline\":  \"MITRE IDs appear inline in NL prose (1=yes, as required by training)\",\n    \"graph_has_arrows\":     \"Attack graph section has at least 1 -> arrow (1=yes)\",\n}\n\n# Which direction is 'better' for each metric (used in report)\nHIGHER_IS_BETTER = {\n    \"sections_found\", \"sections_in_order\", \"all_sections_present\",\n    \"mitre_recall\", \"mitre_precision\", \"g_notation_exact\",\n    \"g_components_defined\", \"arrow_count\", \"numbered_stages\",\n    \"word_count\", \"nl_word_count\", \"distinct_1\", \"distinct_2\",\n    \"type_token_ratio\", \"nl_has_mitre_inline\", \"graph_has_arrows\",\n}\nLOWER_IS_BETTER = {\"markdown_violations\", \"output_truncated\"}\n\nNUMERIC_METRICS = [\n    \"sections_found\", \"sections_in_order\", \"all_sections_present\",\n    \"mitre_recall\", \"mitre_precision\",\n    \"g_notation_exact\", \"g_components_defined\", \"arrow_count\",\n    \"numbered_stages\", \"markdown_violations\",\n    \"word_count\", \"nl_word_count\", \"distinct_1\", \"distinct_2\",\n    \"output_truncated\", \"nl_has_mitre_inline\", \"graph_has_arrows\",\n]\n\n\ndef _avg(values: list) -> float:\n    valid = [float(v) for v in values if v != \"\" and v is not None]\n    return round(sum(valid) / len(valid), 4) if valid else 0.0\n\n\ndef print_table(all_metrics: list[OutputMetrics]):\n    \"\"\"Print a compact per-output metrics table to stdout.\"\"\"\n    col_w = 22\n    id_w  = 10\n    m_w   = 9\n\n    header = f\"{'model':>10} {'pid':>{id_w}}\"\n    for m in NUMERIC_METRICS:\n        header += f\" {m[:m_w]:>{m_w}}\"\n    print(header)\n    print(\"-\" * len(header))\n\n    for om in all_metrics:\n        row = f\"{om.model:>10} {om.prompt_id:>{id_w}}\"\n        for m in NUMERIC_METRICS:\n            val = getattr(om, m)\n            row += f\" {str(val):>{m_w}}\"\n        print(row)\n\n\ndef aggregate_by_model(all_metrics: list[OutputMetrics]) -> dict[str, dict]:\n    agg = {}\n    for model_name in (\"baseline\", \"finetuned\"):\n        subset = [om for om in all_metrics if om.model == model_name]\n        if not subset:\n            continue\n        agg[model_name] = {}\n        for m in NUMERIC_METRICS:\n            vals = [getattr(om, m) for om in subset]\n            agg[model_name][m] = _avg(vals)\n    return agg\n\n\ndef write_markdown_report(\n    all_metrics: list[OutputMetrics],\n    agg: dict[str, dict],\n    out_path: Path\n):\n    lines = []\n    lines.append(\"# Scenario Generator — Manual Evaluation Report\")\n    lines.append(\"\")\n    lines.append(\n        \"> **Methodology**: Reference-free metrics. No LLM judge used.  \\n\"\n        \"> Each metric directly tests a rule the fine-tuned model was trained on.\"\n    )\n    lines.append(\"\")\n\n    # ── Metric legend ──\n    lines.append(\"## Metric Definitions\")\n    lines.append(\"\")\n    lines.append(\"| Metric | What it measures | Better direction |\")\n    lines.append(\"|---|---|:---:|\")\n    for m in NUMERIC_METRICS:\n        direction = \"↑ higher\" if m in HIGHER_IS_BETTER else \"↓ lower\"\n        lines.append(f\"| `{m}` | {METRIC_EXPLANATIONS.get(m, '')} | {direction} |\")\n    lines.append(\"\")\n\n    # ── Aggregate comparison ──\n    lines.append(\"## Baseline vs Fine-Tuned: Averaged Scores\")\n    lines.append(\"\")\n    lines.append(\"| Metric | Baseline | Fine-Tuned | Δ | Winner |\")\n    lines.append(\"|---|---:|---:|---:|:---:|\")\n    for m in NUMERIC_METRICS:\n        b = agg.get(\"baseline\",  {}).get(m, \"—\")\n        f = agg.get(\"finetuned\", {}).get(m, \"—\")\n        if isinstance(b, float) and isinstance(f, float):\n            delta = round(f - b, 4)\n            if m in HIGHER_IS_BETTER:\n                winner = \"🟢 FT\" if delta > 0 else (\"🔴 Base\" if delta < 0 else \"Tie\")\n            else:\n                winner = \"🟢 FT\" if delta < 0 else (\"🔴 Base\" if delta > 0 else \"Tie\")\n            delta_str = f\"+{delta}\" if delta > 0 else str(delta)\n        else:\n            delta_str, winner = \"—\", \"—\"\n        lines.append(f\"| `{m}` | {b} | {f} | {delta_str} | {winner} |\")\n    lines.append(\"\")\n\n    # ── Per-prompt breakdown ──\n    lines.append(\"## Per-Prompt Breakdown\")\n    lines.append(\"\")\n    prompt_ids = sorted(set(om.prompt_id for om in all_metrics))\n    for pid in prompt_ids:\n        pair = {om.model: om for om in all_metrics if om.prompt_id == pid}\n        b_om = pair.get(\"baseline\")\n        f_om = pair.get(\"finetuned\")\n        if b_om:\n            lines.append(f\"### {pid}: {b_om.focus}\")\n        lines.append(\"\")\n        lines.append(\"| Metric | Baseline | Fine-Tuned |\")\n        lines.append(\"|---|---:|---:|\")\n        for m in NUMERIC_METRICS:\n            b_val = getattr(b_om, m, \"—\") if b_om else \"—\"\n            f_val = getattr(f_om, m, \"—\") if f_om else \"—\"\n            lines.append(f\"| `{m}` | {b_val} | {f_val} |\")\n        lines.append(\"\")\n\n    # ── Interpretation guide ──\n    lines.append(\"## How to Interpret Results\")\n    lines.append(\"\")\n    lines.append(\"| What you want to see | Why |\")\n    lines.append(\"|---|---|\")\n    lines.append(\"| Fine-tuned `sections_in_order = 1` on all prompts | Training enforced strict section ordering |\")\n    lines.append(\"| Fine-tuned `mitre_recall` ≥ baseline | Fine-tuned should reference all input technique IDs |\")\n    lines.append(\"| Fine-tuned `mitre_precision` ≥ baseline | Fine-tuned should not hallucinate MITRE IDs not in input |\")\n    lines.append(\"| Fine-tuned `g_notation_exact = 1` | Exact G=(N,S,V,P,FS) notation was in every training sample |\")\n    lines.append(\"| Fine-tuned `g_components_defined` = 5 | All 5 components should be separately defined |\")\n    lines.append(\"| Fine-tuned `markdown_violations = 0` | Training explicitly forbade **, ##, etc. |\")\n    lines.append(\"| Fine-tuned `nl_has_mitre_inline = 1` | Training required technique IDs inline in NL prose |\")\n    lines.append(\"| Fine-tuned `graph_has_arrows = 1` | Training required -> arrows in attack graph |\")\n    lines.append(\"| Fine-tuned `output_truncated = 0` | Output should reach a natural conclusion |\")\n    lines.append(\"\")\n\n    out_path.write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n    print(f\"[OK] Markdown report saved: {out_path}\")\n\n\ndef write_csv(all_metrics: list[OutputMetrics], out_path: Path):\n    field_names = [f.name for f in fields(OutputMetrics)]\n    with out_path.open(\"w\", newline=\"\", encoding=\"utf-8\") as fp:\n        w = csv.DictWriter(fp, fieldnames=field_names)\n        w.writeheader()\n        for om in all_metrics:\n            w.writerow({f.name: getattr(om, f.name) for f in fields(OutputMetrics)})\n    print(f\"[OK] CSV saved: {out_path}\")\n\n\n# ── Main ──────────────────────────────────────────────────────────────────────\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Reference-free eval of prompt_gen_comparison.txt\")\n    parser.add_argument(\"--txt\",  required=True, help=\"Path to prompt_gen_comparison.txt\")\n    parser.add_argument(\"--csv\",  action=\"store_true\", help=\"Also save eval_results.csv\")\n    parser.add_argument(\"--md\",   default=\"\", help=\"Path for markdown report (default: eval_report.md next to --txt)\")\n    args = parser.parse_args()\n\n    txt_path = Path(args.txt)\n    if not txt_path.exists():\n        print(f\"[!] File not found: {txt_path}\")\n        sys.exit(1)\n\n    out_dir  = txt_path.parent\n    md_path  = Path(args.md) if args.md else out_dir / \"eval_report.md\"\n    csv_path = out_dir / \"eval_results.csv\"\n\n    print(f\"[*] Parsing: {txt_path}\")\n    records = parse_comparison_txt(txt_path)\n    if not records:\n        print(\"[!] No output records found. Check the TXT format.\")\n        sys.exit(1)\n    print(f\"[OK] Found {len(records)} outputs ({len(records)//2} prompts x 2 models)\")\n\n    print(\"[*] Computing metrics...\")\n    all_metrics = [compute_metrics(r) for r in records]\n\n    print()\n    print_table(all_metrics)\n\n    agg = aggregate_by_model(all_metrics)\n\n    print()\n    print(\"== AVERAGE SCORES ===============================================\")\n    print(f\"{'metric':<30} {'baseline':>10} {'finetuned':>10}\")\n    print(\"-\" * 52)\n    for m in NUMERIC_METRICS:\n        b = agg.get(\"baseline\",  {}).get(m, \"-\")\n        f = agg.get(\"finetuned\", {}).get(m, \"-\")\n        print(f\"  {m:<28} {str(b):>10} {str(f):>10}\")\n\n    write_markdown_report(all_metrics, agg, md_path)\n    if args.csv:\n        write_csv(all_metrics, csv_path)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding='utf-8')
pathlib.Path("attack_paths.json").write_text("[\n  {\n    \"source_id\": \"AP-001\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> InsecureS3Bucket\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account, T1486 - Data Encrypted for Impact, T1485 - Data Destruction, T1490 - Inhibit System Recovery\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Ransomware or destruction\"\n  },\n  {\n    \"source_id\": \"AP-002\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> PublicRDS\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1078 - Valid Accounts, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1486 - Data Encrypted for Impact, T1485 - Data Destruction, T1490 - Inhibit System Recovery\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Ransomware or destruction\"\n  },\n  {\n    \"source_id\": \"AP-003\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> InsecureS3Bucket\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Data theft\"\n  },\n  {\n    \"source_id\": \"AP-004\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> PublicRDS\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1078 - Valid Accounts, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Data theft\"\n  },\n  {\n    \"source_id\": \"AP-005\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> PublicRDS\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1078 - Valid Accounts, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1498 - Network Denial of Service, T1499 - Endpoint Denial of Service, T1489 - Service Stop\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Denial of service\"\n  },\n  {\n    \"source_id\": \"AP-006\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1098 - Account Manipulation, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1531 - Account Access Removal\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model\",\n    \"objective\": \"Identity takeover\"\n  },\n  {\n    \"source_id\": \"AP-007\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> InsecureS3Bucket\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account, T1486 - Data Encrypted for Impact, T1485 - Data Destruction, T1490 - Inhibit System Recovery\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Ransomware or destruction\"\n  },\n  {\n    \"source_id\": \"AP-008\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> PublicRDS\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1078 - Valid Accounts, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1486 - Data Encrypted for Impact, T1485 - Data Destruction, T1490 - Inhibit System Recovery\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Ransomware or destruction\"\n  },\n  {\n    \"source_id\": \"AP-009\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> InsecureS3Bucket\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Data theft\"\n  },\n  {\n    \"source_id\": \"AP-010\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> PublicRDS\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1078 - Valid Accounts, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Data theft\"\n  },\n  {\n    \"source_id\": \"AP-011\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> ReportsTable\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1486 - Data Encrypted for Impact, T1485 - Data Destruction, T1490 - Inhibit System Recovery\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Ransomware or destruction\"\n  },\n  {\n    \"source_id\": \"AP-012\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance -> OverPermissiveInstanceProfile -> OverPermissiveRole -> ReportsTable\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Data theft\"\n  },\n  {\n    \"source_id\": \"AP-013\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> PublicRDS\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1078 - Valid Accounts, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1498 - Network Denial of Service, T1499 - Endpoint Denial of Service, T1489 - Service Stop\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Denial of service\"\n  },\n  {\n    \"source_id\": \"AP-014\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"CRITICAL\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021.007 - Remote Services: Cloud Services, T1531 - Account Access Removal\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model\",\n    \"objective\": \"Identity takeover\"\n  },\n  {\n    \"source_id\": \"AP-015\",\n    \"source_name\": \"OpenSecurityGroup -> PublicEC2Instance\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"HIGH\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021 - Remote Services, T1021.007 - Remote Services: Cloud Services, T1498 - Network Denial of Service, T1499 - Endpoint Denial of Service, T1489 - Service Stop\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment\",\n    \"objective\": \"Denial of service\"\n  },\n  {\n    \"source_id\": \"AP-016\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> ReportsTable\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"HIGH\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021.007 - Remote Services: Cloud Services, T1486 - Data Encrypted for Impact, T1485 - Data Destruction, T1490 - Inhibit System Recovery\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Ransomware or destruction\"\n  },\n  {\n    \"source_id\": \"AP-017\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole -> OverPermissiveRole -> ReportsTable\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"HIGH\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1199 - Trusted Relationship, T1648 - Serverless Execution, T1098 - Account Manipulation, T1098.001 - Account Manipulation: Additional Cloud Credentials, T1098.003 - Account Manipulation: Additional Cloud Roles, T1548.005 - Abuse Elevation Control Mechanism: Temporary Elevated Cloud Access, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1619 - Cloud Storage Object Discovery, T1021.007 - Remote Services: Cloud Services, T1530 - Data from Cloud Storage, T1537 - Transfer Data to Cloud Account\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model, File system / cloud storage / databases\",\n    \"objective\": \"Data theft\"\n  },\n  {\n    \"source_id\": \"AP-018\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance -> AppInstanceProfile -> AppServerRole -> DeploymentRole\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"HIGH\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1648 - Serverless Execution, T1098 - Account Manipulation, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021.007 - Remote Services: Cloud Services, T1531 - Account Access Removal\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment, OS / IAM permission model\",\n    \"objective\": \"Identity takeover\"\n  },\n  {\n    \"source_id\": \"AP-019\",\n    \"source_name\": \"AppSecurityGroup -> AppServerInstance\",\n    \"industry\": \"Information technology service activities (ISIC J62)\",\n    \"attack_level\": \"HIGH\",\n    \"techniques\": \"T1190 - Exploit Public-Facing Application, T1078.004 - Valid Accounts: Cloud Accounts, T1552.005 - Unsecured Credentials: Cloud Instance Metadata API, T1580 - Cloud Infrastructure Discovery, T1021.007 - Remote Services: Cloud Services, T1498 - Network Denial of Service, T1499 - Endpoint Denial of Service, T1489 - Service Stop\",\n    \"surfaces\": \"Internet-facing services / user endpoints, Application runtime environment\",\n    \"objective\": \"Denial of service\"\n  }\n]\n", encoding='utf-8')
print('Wrote:', "compare_models.py", "prompt_generator.py", "prompt_format.py", "data_split.py", "judge_eval.py", "manual_eval_metrics.py", "attack_paths.json")

In [ ]:
OUT_DIR = '/kaggle/working/model_comparison'

In [ ]:
import os
REQUIRED = ['adapter_config.json', 'adapter_model.safetensors']
def find_adapter_dir(base='/kaggle/input'):
    for root, _dirs, fs in os.walk(base):
        if all(r in fs for r in REQUIRED):
            return root
    return None
ADAPTER_DIR = find_adapter_dir()
print('Adapter dir:', ADAPTER_DIR)
assert ADAPTER_DIR, 'Attach the finetuned adapter dataset under Input.'

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    print('Judge key loaded.')
except Exception as e:
    print('No OPENROUTER_API_KEY secret, so only format metrics will be reported:', e)

In [ ]:
!python compare_models.py --adapter_path "{ADAPTER_DIR}" --attack_paths attack_paths.json --out_dir "{OUT_DIR}"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(OUT_DIR, 'report.md'), encoding='utf-8').read()))